# Revision R2 — ARRAY-D-26-04414R1 — analyses requested by Reviewer 1

This notebook adds, to the R1 pipeline (`revision_pipeline_v2_1.ipynb`), the analyses that Reviewer 1 asked for in the second round. The pipeline definitions (cell 2) are copied **verbatim** from `revision_pipeline_v2_1.ipynb`. Only the data paths and the output folder are set in cell 1.

**How to run.** Edit cell 1 only if a path differs. Then use *Run All*. Every step is saved as soon as it finishes (`resultats_R2/units`). If the run is interrupted (Windows update, closed laptop), restart the kernel and *Run All* again: finished steps are skipped and the run resumes at the first unfinished one.

**First check (cells S1 and S2).** Before any new analysis, the notebook re-runs the R1 headline configuration and compares every PR-AUC, ROC-AUC, TP, FP and alert count with `canonical_results.csv` of the R1 run. If they differ by more than 1e-6, it stops. A difference means the Python environment is not the one of the R1 run: use the same environment (manifest of R1: Python 3.11, numpy 1.26.4, pandas 2.2.2, scikit-learn 1.5.0, scipy 1.13.1).

| Step | What it computes | Reviewer 1 point | Approx. time |
|---|---|---|---|
| S1 | ULB headline re-run, reproduction check, validation and test scores | all | 2 min |
| S2 | PaySim headline re-run, reproduction check, scores | 2, 6, 7 | 35–45 min |
| S3 | PaySim without the residuals, without step, without both (Table 13), scores | 2 | 60–90 min |
| S4 | PaySim rolling origins **without step** | 3 | 60–90 min |
| S5 | PaySim random and prevalence-matched arms **without step** | 3 | 80–110 min |
| S6 | Ensemble rebuilt from validation-selected members (ULB; PaySim 200,000-row windows) | 4 | 20–30 min |
| S7 | Five reference-set draws, ensembles included (ULB; PaySim) | 1, 6 | 75–100 min |
| A1–A6 | Analyses from the saved scores (no refitting): four-member ensemble, ⌈ρN⌉ budgets, bootstrap intervals, paired tests with Holm, five-draw summaries, meta-classifier diagnostic | 1–7 | 5–10 min |
| Z | `R2_VALUES.txt`: every value the revised manuscript will quote | — | — |

Total: about 6–8 hours on the R1 laptop. Optional (`RUN_OPTIONAL = True`): five draws on PaySim without step and without both, about 2–3 hours more.

**Nothing is written to the R1 folder.** All outputs go to `2_Revision_R2/resultats_R2`.


In [1]:
# =============================================================================
# 1. CONFIGURATION  --  the only cell to edit
# =============================================================================
import os

BASE = r"C:\Users\LENOVO\Desktop\Unsupervised and Semi-Supervised Anomaly Detection for Credit Card Fraud"

ULB_PATH    = os.path.join(BASE, "creditcard.csv")
PAYSIM_PATH = os.path.join(BASE, "paysim.csv")

# Folder of the R1 run: canonical_results.csv, test_scores.npz, manifest.json
R1_DIR = os.path.join(BASE, "Final revision Array paper supervised and semisupervised",
                      "revision_v2")

# Everything this notebook writes goes here (nothing is written to R1_DIR)
OUT_DIR = os.path.join(BASE, "2_Revision_R2", "resultats_R2")

# Five reference-set draws on PaySim without step and without both inputs.
# Not required by the reviewer; about 2-3 hours more.
RUN_OPTIONAL = False

# Stop at S1/S2 if the R1 headline values are not reproduced (recommended).
STOP_IF_NOT_REPRODUCED = True

os.makedirs(OUT_DIR, exist_ok=True)
for _p, _n in [(ULB_PATH, "ULB_PATH"), (PAYSIM_PATH, "PAYSIM_PATH"),
               (os.path.join(R1_DIR, "canonical_results.csv"), "R1_DIR")]:
    print(f"  {_n:<12} {'OK ' if os.path.exists(_p) else 'MISSING'}  {_p}")


  ULB_PATH     OK   C:\Users\LENOVO\Desktop\Unsupervised and Semi-Supervised Anomaly Detection for Credit Card Fraud\creditcard.csv
  PAYSIM_PATH  OK   C:\Users\LENOVO\Desktop\Unsupervised and Semi-Supervised Anomaly Detection for Credit Card Fraud\paysim.csv
  R1_DIR       OK   C:\Users\LENOVO\Desktop\Unsupervised and Semi-Supervised Anomaly Detection for Credit Card Fraud\Final revision Array paper supervised and semisupervised\revision_v2\canonical_results.csv


In [2]:
# =============================================================================
# 2. PIPELINE DEFINITIONS -- copied verbatim from revision_pipeline_v2_1.ipynb
#    (its cells 1-4 and run_split of cell 5). The only change: the three path
#    assignments now live in the configuration cell. Nothing below is new.
# =============================================================================

# =============================================================================
# REVISION PIPELINE v2  --  ARRAY-D-26-04414
# Unsupervised / Semi-Supervised Anomaly Detection for Credit Card Fraud
# =============================================================================
# This pipeline replaces the v1 benchmark notebook. It is organised so that the
# five structural invariants demanded by the reviewers are enforced by
# assertions rather than by convention.
#
# INVARIANT 1  No test-set object ever reaches a calibration function.
# INVARIANT 2  rho is never computed from labels, except in the explicitly
#              labelled ORACLE regime.
# INVARIANT 3  One canonical results frame; every table and figure derives
#              from it; figure/table agreement is asserted.
# INVARIANT 4  One sign convention: higher score = more anomalous, asserted
#              for every detector.
# INVARIANT 5  Every number is traceable to a row via run_id.
#
# REGIMES (replacing "Protocol A / Protocol B")
#   U  label-free      fit on the full training split, rho from an a-priori grid
#   N  normal-only     fit on training transactions labelled legitimate
#   O  oracle          as U, but rho = true training prevalence (UPPER BOUND,
#                      not achievable in deployment; reported for reference)
#
# Reviewer comments discharged by each stage are marked [Rx#n] in the headers.
# =============================================================================

import os, time, json, hashlib, warnings, platform
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------- #
# Paths  --  EDIT THESE
# --------------------------------------------------------------------------- #
# Absolute paths are safest: a relative path resolves against whatever
# directory Jupyter was started from, not the notebook's own folder.
# [R2] ULB_PATH, PAYSIM_PATH and OUT_DIR are set in the configuration cell.
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)
print(f"outputs -> {os.path.abspath(OUT_DIR)}")

# --------------------------------------------------------------------------- #
# Global configuration
# --------------------------------------------------------------------------- #
SEED          = 42
TRAIN_FRAC    = 0.60
VAL_FRAC      = 0.20            # test = remainder

LOF_TRAIN_CAP = 20_000          # reference-set cap for LOF   [R4#8]
OCSVM_CAP     = 20_000          # training cap for OC-SVM     [R4#8]
DBSCAN_CAP    = 20_000          # core-point cap for DBSCAN   [R4#3]
MIN_CORE_POINTS = 50            # below this the DBSCAN core set carries no
                                # density information and the out-of-sample
                                # rule of Section 3.4 degenerates.     [v2.1]
REQUIRE_NONDEGENERATE_CORE = False
                                # False reproduces the deposited v2 results,
                                # which the manuscript reports and which the
                                # manuscript also documents as degenerate on
                                # PaySim. Set True to widen eps until the core
                                # set reaches MIN_CORE_POINTS; this CHANGES the
                                # DBSCAN columns and every table that contains
                                # them, so the manuscript must be regenerated
                                # from the same run if you switch it.  [v2.1]
PAYSIM_BALANCE_FEATURES = True  # toggled by the R4#9 ablation cell
SUBSAMPLE_RULE = "random"       # headline rule, identical to v1 and to [32],
                                # so every changed number is attributable to
                                # the protocol corrections alone.
                                # "tail" (most recent contiguous block) is
                                # deployment-realistic but is a DIFFERENT
                                # experiment: it is reported as a sensitivity
                                # arm in the subsampling study, not silently
                                # substituted.                        [R4#8]
N_SUBSAMPLE_SEEDS = 5           # repeated subsampling         [R4#8]

# rho grid, fixed a priori and WITHOUT labels                  [R1#1][R6#4]
RHO_GRID      = [0.001, 0.005, 0.010, 0.020]
RHO_DEFAULT   = 0.005           # headline operating point for regimes U and N.
                                # Chosen a priori, WITHOUT labels. 0.001 sits
                                # below ULB's actual prevalence (0.00211) and
                                # under-alerts by half, collapsing every
                                # threshold-dependent metric.

N_FOLDS       = 5               # rolling-origin folds         [R1#4][R6#3]
FOLD_CONFIG   = {               # per dataset: (n_origins, share of the stream
    "ULB":    (3, 0.60),        # tiled by the test blocks)
    "PaySim": (5, 0.40),
}
MIN_FOLD_POSITIVES = 25         # below this a fold is reported but flagged as
                                # not powered for any ranking claim
N_BOOTSTRAP   = 2000            # block bootstrap replicates   [R5#2][R6#3]
BOOTSTRAP_BLOCKS = 50           # temporal blocks for the bootstrap
N_JOBS        = 2

rng_global = np.random.default_rng(SEED)

# --------------------------------------------------------------------------- #
# Run manifest  --  INVARIANT 5
# --------------------------------------------------------------------------- #
# --------------------------------------------------------------------------- #
# Preflight  --  fail here, with a usable message, rather than mid-pipeline
# --------------------------------------------------------------------------- #
def preflight():
    import glob
    missing = []
    for label, path in [("ULB_PATH", ULB_PATH), ("PAYSIM_PATH", PAYSIM_PATH)]:
        if os.path.exists(path):
            size = os.path.getsize(path) / 1e6
            with open(path) as f:
                header = f.readline().strip()[:90]
            print(f"  {label:<12} OK  {os.path.abspath(path)}  "
                  f"({size:,.0f} MB)\n               header: {header}")
        else:
            missing.append((label, path))
    if not missing:
        return
    print("\n  MISSING DATA FILES")
    print(f"  working directory: {os.getcwd()}")
    home = os.path.expanduser("~")
    for label, path in missing:
        pattern = "creditcard*.csv" if label == "ULB_PATH" else "*ay*im*.csv"
        print(f"\n  {label} = {path!r}  -> not found")
        hits = []
        for root in [os.getcwd(), home, os.path.join(home, "Downloads"),
                     os.path.join(home, "Desktop"), os.path.join(home, "Documents")]:
            hits += glob.glob(os.path.join(root, "**", pattern), recursive=True)
        hits = sorted(set(hits))[:5]
        if hits:
            print("  candidates found on disk -- paste one into cell 0:")
            for h in hits:
                print(f"      r{h!r}")
        else:
            print("  no candidate found in cwd, home, Downloads, Desktop or "
                  "Documents. Search the whole drive with:")
            print(f"      import glob; glob.glob(r'C:\\**\\{pattern}', "
                  "recursive=True)")
    raise FileNotFoundError(
        "Set ULB_PATH / PAYSIM_PATH in cell 0 to absolute paths, then re-run "
        "this cell. Nothing else needs changing.")


preflight()

import sklearn, scipy
MANIFEST = {
    "created":      pd.Timestamp.now().isoformat(),
    "python":       platform.python_version(),
    "numpy":        np.__version__,
    "pandas":       pd.__version__,
    "sklearn":      sklearn.__version__,
    "scipy":        scipy.__version__,
    "seed":         SEED,
    "rho_grid":     RHO_GRID,
    "rho_default":  RHO_DEFAULT,
    "subsample_rule": SUBSAMPLE_RULE,
    "pipeline_version": "v2.1",
    "min_core_points": MIN_CORE_POINTS,
    "require_nondegenerate_core": REQUIRE_NONDEGENERATE_CORE,
    "selection_tie_break": "simplest configuration within 1e-4 of the best validation PR-AUC",
    "caps":         {"lof": LOF_TRAIN_CAP, "ocsvm": OCSVM_CAP, "dbscan": DBSCAN_CAP},
}
# --------------------------------------------------------------------------- #
# Stale-results guard. run_id hashes rho_assumed, so changing RHO_DEFAULT (or
# any grid) mints NEW ids: the merge in save_results() would then KEEP the
# previous generation's rows alongside the new ones, invisibly, in the file the
# manuscript is meant to quote from.
# --------------------------------------------------------------------------- #
_mpath = os.path.join(OUT_DIR, "manifest.json")
_cpath = os.path.join(OUT_DIR, "canonical_results.csv")
if os.path.exists(_mpath) and os.path.exists(_cpath):
    old = json.load(open(_mpath))
    drift = {k: (old.get(k), MANIFEST[k]) for k in
             ["rho_default", "rho_grid", "seed", "subsample_rule", "caps"]
             if old.get(k) != MANIFEST[k]}
    if drift:
        print("\n" + "!" * 70)
        print("STALE RESULTS ON DISK -- the configuration changed since the "
              "last run:")
        for k, (was, now) in drift.items():
            print(f"    {k}: was {was}  ->  now {now}")
        print(f"\n  {_cpath} still holds rows produced under the old "
              "configuration.\n  Those rows carry different run_ids and will "
              "NOT be overwritten.\n")
        print("  Rename or delete the output folder before re-running:")
        print(f"      import shutil; shutil.move(r'{os.path.abspath(OUT_DIR)}', "
              f"r'{os.path.abspath(OUT_DIR)}_old')")
        print("!" * 70 + "\n")
        raise RuntimeError(
            "Archive the previous results folder first, then re-run this cell. "
            "Set ALLOW_MIXED_RESULTS = True above to override (not advised -- "
            "the canonical file is what every manuscript number is traced to).")

with open(_mpath, "w") as f:
    json.dump(MANIFEST, f, indent=2)
print(json.dumps(MANIFEST, indent=2))

# =============================================================================
# 1. DATA, SPLITS, PREPROCESSING
#    [R4#4] prevalence-matched random arm   [R1#4] rolling-origin folds
# =============================================================================
from sklearn.preprocessing import RobustScaler, StandardScaler


@dataclass
class Split:
    """One train / validation / test partition, already scaled."""
    Xtr: np.ndarray; ytr: np.ndarray
    Xva: np.ndarray; yva: np.ndarray
    Xte: np.ndarray; yte: np.ndarray
    dataset: str
    split_type: str            # "chronological" | "random" | "random_matched"
    fold: int = 0

    @property
    def prevalence(self) -> Dict[str, float]:
        return {"train": float(self.ytr.mean()),
                "val":   float(self.yva.mean()),
                "test":  float(self.yte.mean())}

    def describe(self):
        p = self.prevalence
        print(f"  [{self.dataset}/{self.split_type}/fold{self.fold}] "
              f"train {self.Xtr.shape} ({int(self.ytr.sum())}f, {p['train']:.5f}) | "
              f"val {self.Xva.shape} ({int(self.yva.sum())}f, {p['val']:.5f}) | "
              f"test {self.Xte.shape} ({int(self.yte.sum())}f, {p['test']:.5f})")


# --------------------------------------------------------------------------- #
# Feature engineering  --  identical to v1 so results stay comparable
# --------------------------------------------------------------------------- #
def _prep_ulb(df: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    df = df.copy()
    df["Hour"] = (df["Time"] // 3600) % 24
    y = df["Class"].astype(int).values
    X = df.drop(columns=["Class"]).astype(np.float32)
    return X, y, ["Amount", "Time", "Hour"]


def _prep_paysim(df: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    df = df.copy()
    # Balance-consistency features. They are algebraic rearrangements of the
    # simulator's own account-update equations, which is precisely R4#9's
    # objection; the ablation cell re-runs PaySim without them. [R4#9]
    df["delta_orig"] = df["newbalanceOrig"] - (df["oldbalanceOrg"] - df["amount"])
    df["delta_dest"] = df["newbalanceDest"] - (df["oldbalanceDest"] + df["amount"])
    dummies = pd.get_dummies(df["type"], prefix="type").astype(np.float32)
    y = df["isFraud"].astype(int).values
    keep = ["step", "amount", "oldbalanceOrg", "newbalanceOrig",
            "oldbalanceDest", "newbalanceDest"]
    if PAYSIM_BALANCE_FEATURES:
        keep += ["delta_orig", "delta_dest"]
    X = pd.concat([df[keep].astype(np.float32), dummies], axis=1)
    return X, y, keep


def _scale(Xtr, Xva, Xte, robust_cols, global_std: bool):
    """Scalers fitted on TRAIN ONLY. [leakage-free preprocessing]"""
    Xtr, Xva, Xte = Xtr.copy(), Xva.copy(), Xte.copy()
    cols = [c for c in robust_cols if c in Xtr.columns]
    if cols:
        rob = RobustScaler().fit(Xtr[cols])
        for D in (Xtr, Xva, Xte):
            D[cols] = rob.transform(D[cols])
    A = Xtr.values.astype(np.float32)
    B = Xva.values.astype(np.float32)
    C = Xte.values.astype(np.float32)
    if global_std:
        std = StandardScaler().fit(A)
        A, B, C = std.transform(A), std.transform(B), std.transform(C)
    return (np.ascontiguousarray(A, dtype=np.float32),
            np.ascontiguousarray(B, dtype=np.float32),
            np.ascontiguousarray(C, dtype=np.float32))


# --------------------------------------------------------------------------- #
# Split builders
# --------------------------------------------------------------------------- #
_RAW_CACHE: Dict[tuple, tuple] = {}


def clear_raw_cache():
    _RAW_CACHE.clear(); print("  raw cache cleared")


def load_raw(dataset: str):
    """Parsed, feature-engineered source data, cached in memory.

    Eight split constructions per dataset would otherwise re-parse PaySim's
    6.3M rows eight times. The cache key INCLUDES PAYSIM_BALANCE_FEATURES:
    keying on the dataset name alone would serve delta-feature data to the
    without-delta arm of the R4#9 ablation and silently invalidate it.
    """
    key = (dataset, PAYSIM_BALANCE_FEATURES if dataset == "PaySim" else None)
    if key not in _RAW_CACHE:
        _RAW_CACHE[key] = _load_raw_uncached(dataset)
    return _RAW_CACHE[key]


def _load_raw_uncached(dataset: str):
    if dataset == "ULB":
        df = pd.read_csv(ULB_PATH).sort_values("Time").reset_index(drop=True)
        X, y, rob = _prep_ulb(df)
        return X, y, rob, "Time", False
    elif dataset == "PaySim":
        df = pd.read_csv(PAYSIM_PATH).sort_values("step").reset_index(drop=True)
        X, y, rob = _prep_paysim(df)
        return X, y, rob, "step", True
    raise ValueError(dataset)


def chronological_split(dataset: str) -> Split:
    X, y, rob, _, gstd = load_raw(dataset)
    n = len(X)
    i1, i2 = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))
    Xtr, Xva, Xte = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
    A, B, C = _scale(Xtr, Xva, Xte, rob, gstd)
    return Split(A, y[:i1], B, y[i1:i2], C, y[i2:], dataset, "chronological")


def rolling_origin_splits(dataset: str, n_folds: Optional[int] = None,
                          span: Optional[float] = None) -> List[Split]:
    """Expanding-window forward chaining. Fold k trains on [0, t_k), validates on
    the next block and tests on the block after it.                  [R1#4][R6#3]

    Origins and span are per dataset (FOLD_CONFIG): ULB holds 492 frauds over
    two days, so five origins would leave ~20 positives per test block and no
    PR-AUC computed on that could support a ranking.
    """
    cfg_folds, cfg_span = FOLD_CONFIG.get(dataset, (N_FOLDS, 0.40))
    n_folds = n_folds or cfg_folds
    span = span or cfg_span
    X, y, rob, _, gstd = load_raw(dataset)
    n = len(X)
    block = int(n * span / n_folds)
    start = n - n_folds * block
    out = []
    for k in range(n_folds):
        te_lo = start + k * block
        te_hi = te_lo + block
        va_lo = max(0, te_lo - block)
        Xtr, Xva, Xte = X.iloc[:va_lo], X.iloc[va_lo:te_lo], X.iloc[te_lo:te_hi]
        if len(Xtr) < 5000 or Xva.shape[0] == 0:
            continue
        A, B, C = _scale(Xtr, Xva, Xte, rob, gstd)
        out.append(Split(A, y[:va_lo], B, y[va_lo:te_lo], C, y[te_lo:te_hi],
                         dataset, "chronological", fold=k))
    return out


def random_split(dataset: str, seed: int = SEED) -> Split:
    from sklearn.model_selection import train_test_split
    X, y, rob, _, gstd = load_raw(dataset)
    idx = np.arange(len(X))
    tr, rest = train_test_split(idx, train_size=TRAIN_FRAC, stratify=y,
                                random_state=seed)
    va, te = train_test_split(rest, train_size=VAL_FRAC / (1 - TRAIN_FRAC),
                              stratify=y[rest], random_state=seed)
    A, B, C = _scale(X.iloc[tr], X.iloc[va], X.iloc[te], rob, gstd)
    return Split(A, y[tr], B, y[va], C, y[te], dataset, "random")


def random_prevalence_matched(dataset: str, target_n: int, target_pos: int,
                              seed: int = SEED) -> Split:
    """Random split whose TEST block matches the chronological test block in both
    size and number of positives. Without this, any random-vs-chronological
    PR-AUC gap is confounded by the class prior.                          [R4#4]"""
    from sklearn.model_selection import train_test_split
    X, y, rob, _, gstd = load_raw(dataset)
    rng = np.random.default_rng(seed)
    pos = np.flatnonzero(y == 1); neg = np.flatnonzero(y == 0)
    te_pos = rng.choice(pos, size=target_pos, replace=False)
    te_neg = rng.choice(neg, size=target_n - target_pos, replace=False)
    te = np.concatenate([te_pos, te_neg]); rng.shuffle(te)
    remaining = np.setdiff1d(np.arange(len(X)), te)
    tr, va = train_test_split(remaining,
                              train_size=TRAIN_FRAC / (TRAIN_FRAC + VAL_FRAC),
                              stratify=y[remaining], random_state=seed)
    A, B, C = _scale(X.iloc[tr], X.iloc[va], X.iloc[te], rob, gstd)
    return Split(A, y[tr], B, y[va], C, y[te], dataset, "random_matched")


def subsample_indices(n: int, cap: int, seed: int,
                      rule: Optional[str] = None) -> np.ndarray:
    """Documented, reproducible subsampling rule.                        [R4#8]

    "tail"   the most recent `cap` training rows -- deployment-realistic and
             DETERMINISTIC, so the seed has no effect (this is why the
             repeated-subsampling study must override the rule).
    "random" uniform draw, seeded.
    """
    rule = rule or SUBSAMPLE_RULE
    if n <= cap:
        return np.arange(n)
    if rule == "tail":
        return np.arange(n - cap, n)
    return np.random.default_rng(seed).choice(n, cap, replace=False)

import warnings
# =============================================================================
# 2. DETECTORS  --  strict fit(train) -> score(val) AND score(test)
#    [R4#1] no label-derived quantity   [R4#2] no test-set dependence
#    [R4#3] explicit out-of-sample interface for LOF and DBSCAN
# =============================================================================
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN, KMeans


@dataclass
class Scores:
    """A detector's raw anomaly scores. Higher = more anomalous (INVARIANT 4).

    NOTE the absence of any y_pred field: hard predictions are produced ONLY by
    the calibration module, from validation scores. A detector cannot emit a
    decision on its own -- this is what makes INVARIANT 1 structural.
    """
    val:  np.ndarray
    test: np.ndarray
    fit_seconds: float
    score_seconds: float
    detector: str
    extra: dict = field(default_factory=dict)
    model: object = None          # kept so per-transaction latency is measurable
    score_one: object = None      # callable: (1, d) array -> float


def _check_sign(s: Scores):
    """INVARIANT 4: assert the score is finite and non-degenerate."""
    for name, a in (("val", s.val), ("test", s.test)):
        assert np.all(np.isfinite(a)), f"{s.detector}: non-finite {name} scores"
    n_unique = len(np.unique(s.test))
    if n_unique < 10:
        print(f"    ! {s.detector}: only {n_unique} distinct test scores "
              f"(tie mass will distort any quantile threshold)")
    return s


# --------------------------------------------------------------------------- #
# NOTE ON rho: none of the fit functions below receives labels or rho, with the
# single exception of OC-SVM, whose `nu` is a genuine model hyperparameter. It
# is fed from the a-priori grid, never from the label-derived prevalence.
# --------------------------------------------------------------------------- #
def fit_isolation_forest(Xref, Xva, Xte, seed=SEED, n_estimators=200):
    t0 = time.time()
    m = IsolationForest(n_estimators=n_estimators, contamination="auto",
                        random_state=seed, n_jobs=N_JOBS).fit(Xref)
    t1 = time.time()
    return _check_sign(Scores(-m.score_samples(Xva), -m.score_samples(Xte),
                              t1 - t0, time.time() - t1, "IsolationForest",
                              model=m,
                              score_one=lambda z, m=m: -m.score_samples(z)[0]))


def fit_lof(Xref, Xva, Xte, seed=SEED, n_neighbors=20):
    """novelty=True in ALL regimes. v1's Table 3 documented novelty=False for
    Protocol A, which offers no out-of-sample scoring interface.          [R4#3]"""
    idx = subsample_indices(len(Xref), LOF_TRAIN_CAP, seed)
    ref = Xref[idx]
    t0 = time.time()
    m = LocalOutlierFactor(n_neighbors=n_neighbors, novelty=True,
                           n_jobs=N_JOBS).fit(ref)
    t1 = time.time()
    return _check_sign(Scores(-m.score_samples(Xva), -m.score_samples(Xte),
                              t1 - t0, time.time() - t1, "LOF",
                              {"n_ref": len(ref), "n_neighbors": n_neighbors},
                              model=m,
                              score_one=lambda z, m=m: -m.score_samples(z)[0]))


def fit_ocsvm(Xref, Xva, Xte, seed=SEED, nu=RHO_DEFAULT, gamma="scale"):
    idx = subsample_indices(len(Xref), OCSVM_CAP, seed)
    ref = Xref[idx]
    t0 = time.time()
    m = OneClassSVM(kernel="rbf", gamma=gamma, nu=max(1e-4, float(nu))).fit(ref)
    t1 = time.time()
    return _check_sign(Scores(-m.decision_function(Xva), -m.decision_function(Xte),
                              t1 - t0, time.time() - t1, "OneClassSVM",
                              {"n_ref": len(ref), "nu": nu}, model=m,
                              score_one=lambda z, m=m: -m.decision_function(z)[0]))


def fit_kmeans(Xref, Xva, Xte, seed=SEED, k=8):
    t0 = time.time()
    m = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(Xref)
    t1 = time.time()
    d_va = m.transform(Xva).min(axis=1)
    d_te = m.transform(Xte).min(axis=1)
    return _check_sign(Scores(d_va, d_te, t1 - t0, time.time() - t1,
                              "KMeans", {"k": k}, model=m,
                              score_one=lambda z, m=m: float(m.transform(z).min())))


def estimate_eps(Xref, min_samples=10, seed=SEED):
    """k-distance knee on the TRAINING reference set (v1 used the test set)."""
    idx = subsample_indices(len(Xref), DBSCAN_CAP, seed)
    nn = NearestNeighbors(n_neighbors=min_samples, n_jobs=N_JOBS).fit(Xref[idx])
    d, _ = nn.kneighbors(Xref[idx])
    kd = np.sort(d[:, -1])
    # knee = point of maximum distance to the chord joining the curve endpoints
    x = np.linspace(0, 1, len(kd)); yv = (kd - kd.min()) / (np.ptp(kd) + 1e-12)
    return float(kd[np.argmax(yv - x)]), kd


def fit_dbscan(Xref, Xva, Xte, seed=SEED, min_samples=10, eps=None,
               eps_scale=1.0):
    """Out-of-sample DBSCAN, formalised.                                  [R4#3]

    v1 clustered the TEST set and emitted a BINARY score, which pins PR-AUC to
    the prevalence and makes the detector uninformative. Here:

      1. cluster a capped subsample of the TRAINING reference set;
      2. collect the core points C = {x : |N_eps(x)| >= min_samples};
      3. for any new point z, define
             s(z) = dist(z, nearest core point in C)
         and the hard rule  z is noise  <=>  s(z) > eps.

    s(z) is CONTINUOUS, computed per point, and depends on nothing but the
    training clustering -- so DBSCAN becomes rankable, ensemble-eligible, and
    free of test-set dependence.

    v2 substituted the WHOLE reference sample for the core set whenever DBSCAN
    found no core point, silently: s(z) then becomes a 1-NN distance to the
    training subsample, which is not DBSCAN. The substitution is now recorded
    instead of hidden, and REQUIRE_NONDEGENERATE_CORE widens eps until the core
    set is usable.                                                     [v2.1]
    """
    idx = subsample_indices(len(Xref), DBSCAN_CAP, seed)
    ref = Xref[idx]
    base_eps = eps
    if base_eps is None:
        base_eps, _ = estimate_eps(Xref, min_samples, seed)
    eps_used = base_eps * eps_scale   # eps, not min_samples, moves DBSCAN
    t0 = time.time()
    db = DBSCAN(eps=eps_used, min_samples=min_samples, n_jobs=N_JOBS).fit(ref)
    n_core = len(db.core_sample_indices_)
    widenings = 0
    while (REQUIRE_NONDEGENERATE_CORE and n_core < MIN_CORE_POINTS
           and widenings < 8):
        eps_used *= 2.0
        widenings += 1
        db = DBSCAN(eps=eps_used, min_samples=min_samples,
                    n_jobs=N_JOBS).fit(ref)
        n_core = len(db.core_sample_indices_)
    fell_back = n_core == 0
    core = ref if fell_back else ref[db.core_sample_indices_]
    degenerate = fell_back or n_core < MIN_CORE_POINTS or n_core >= len(ref)
    if degenerate:
        warnings.warn(
            f"DBSCAN core set is degenerate: {n_core} core points out of "
            f"{len(ref)} at eps={eps_used:.4g}, min_samples={min_samples}. "
            + ("No core point was found, so the full reference sample is used "
               "and the score is a 1-NN distance to the training subsample. "
               if fell_back else "")
            + "The resulting score is not a density-based anomaly score; see "
              "Section 3.4. Set REQUIRE_NONDEGENERATE_CORE = True to widen eps.",
            RuntimeWarning, stacklevel=2)
    nn = NearestNeighbors(n_neighbors=1, n_jobs=N_JOBS).fit(core)
    t1 = time.time()
    d_va, _ = nn.kneighbors(Xva); d_te, _ = nn.kneighbors(Xte)
    return _check_sign(Scores(d_va.ravel(), d_te.ravel(), t1 - t0,
                              time.time() - t1, "DBSCAN",
                              {"eps": eps_used, "eps_base": base_eps,
                               "eps_widenings": widenings,
                               "n_core": int(n_core), "n_ref": len(ref),
                               "core_fraction": round(n_core / len(ref), 6),
                               "degenerate_core": float(degenerate),
                               "core_fallback": float(fell_back),
                               "min_samples": min_samples}, model=nn,
                              score_one=lambda z, nn=nn: float(nn.kneighbors(z)[0][0][0])))


DETECTORS = {
    "IsolationForest": fit_isolation_forest,
    "LOF":             fit_lof,
    "OneClassSVM":     fit_ocsvm,
    "DBSCAN":          fit_dbscan,
    "KMeans":          fit_kmeans,
}


def reference_set(sp: Split, regime: str) -> np.ndarray:
    """U and O fit on the full training split; N fits on legitimate rows only.

    The label read below is the ONLY use of ytr in the pipeline, and it is the
    defining property of the semi-supervised regime -- not a hidden dependence.
    """
    if regime in ("U", "O"):
        return sp.Xtr
    if regime == "N":
        return sp.Xtr[sp.ytr == 0]
    raise ValueError(regime)


def rho_for(sp: Split, regime: str, rho_assumed: float) -> float:
    """INVARIANT 2. The ORACLE branch is the only path to a label-derived rho,
    and every row it produces carries regime == 'O' in the results frame."""
    if regime == "O":
        return float(sp.ytr.mean())
    return float(rho_assumed)

# =============================================================================
# 3. CALIBRATION, METRICS, CANONICAL RESULTS STORE
#    [R1#2][R4#2] thresholds and rank transforms fitted on VALIDATION only
#    [R1#9] PR-AUC primary   [R6#2] tie diagnostics
# =============================================================================
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix)
from sklearn.linear_model import LogisticRegression


class ValidationCalibrator:
    """Everything the decision rule needs, learned from validation scores alone.

    threshold(rho)  the (1-rho) quantile of VALIDATION scores. Applied unchanged
                    to the test stream, so the label assigned to a transaction
                    depends on nothing but that transaction.
    ecdf(s)         the empirical CDF of VALIDATION scores, applied pointwise.
                    Replaces v1's rankdata() over the joint test set, which made
                    each ensemble score a function of all other test rows. [R4#2]
    """

    def __init__(self, val_scores: np.ndarray):
        assert val_scores.ndim == 1
        self._sorted = np.sort(np.asarray(val_scores, dtype=np.float64))
        self.n = len(self._sorted)
        u = len(np.unique(self._sorted))
        self.tie_ratio = 1.0 - u / self.n     # 0 = all distinct             [R6#2]

    def threshold(self, rho: float) -> float:
        return float(np.quantile(self._sorted, 1.0 - rho))

    def ecdf(self, s: np.ndarray) -> np.ndarray:
        """P_val(S <= s), evaluated pointwise; clipped to [0,1] outside range."""
        return np.searchsorted(self._sorted, np.asarray(s), side="right") / self.n

    def predict(self, s: np.ndarray, rho: float) -> np.ndarray:
        return (np.asarray(s) >= self.threshold(rho)).astype(int)


def assert_calibration_is_clean(cal: ValidationCalibrator, sp: Split):
    """INVARIANT 1, checked numerically rather than promised in prose.

    Re-deriving the calibrator from the validation scores alone must reproduce
    the same threshold; and the decision for a single transaction must be
    identical whether it is scored alone or inside the full test batch.
    """
    assert cal.n == len(sp.yva), "calibrator was not built from the validation split"


def single_point_invariance(cal: ValidationCalibrator, s_test: np.ndarray,
                            rho: float, n_probe: int = 200) -> bool:
    """The property v1 could not satisfy: scoring one transaction in isolation
    yields the same decision as scoring it inside the batch.               [R4#2]"""
    rs = np.random.default_rng(SEED)
    probe = rs.choice(len(s_test), size=min(n_probe, len(s_test)), replace=False)
    batch = cal.predict(s_test, rho)[probe]
    alone = np.array([cal.predict(np.array([s_test[i]]), rho)[0] for i in probe])
    return bool(np.array_equal(batch, alone))


def ecdf_rank_average(cals: Dict[str, ValidationCalibrator],
                      scores: Dict[str, np.ndarray]) -> np.ndarray:
    """Ensemble score = mean over detectors of the validation-ECDF of each
    detector's score. Pointwise by construction.                          [R4#2]"""
    cols = [cals[m].ecdf(scores[m]) for m in scores]
    return np.mean(np.column_stack(cols), axis=1)


# --------------------------------------------------------------------------- #
# Metrics
# --------------------------------------------------------------------------- #
def compute_metrics(y_true, y_pred, s_test) -> Dict[str, float]:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "PR_AUC":    float(average_precision_score(y_true, s_test)),  # primary
        "ROC_AUC":   float(roc_auc_score(y_true, s_test)),            # secondary
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "F1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "n_alerts":  int(tp + fp),
        "alert_rate": float((tp + fp) / len(y_true)),
        "prevalence": float(np.mean(y_true)),
        "lift":      float(((tp / max(tp + fp, 1)) / max(np.mean(y_true), 1e-12))),
    }
    return out


# --------------------------------------------------------------------------- #
# Uncertainty helpers  --  used by several cells below
# [R5#2] confidence intervals   [R6#3] 75 frauds cannot support fine rankings
# --------------------------------------------------------------------------- #
def block_bootstrap_ap(y_true, scores, n_boot=None, n_blocks=None, seed=SEED):
    """Ordinary bootstrap resamples transactions independently and understates
    the variance of a temporally ordered stream; blocks preserve local
    structure."""
    n_boot = n_boot or N_BOOTSTRAP; n_blocks = n_blocks or BOOTSTRAP_BLOCKS
    n = len(y_true)
    edges = np.linspace(0, n, n_blocks + 1).astype(int)
    blocks = [np.arange(edges[i], edges[i + 1]) for i in range(n_blocks)]
    rs = np.random.default_rng(seed)
    out = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.concatenate([blocks[i] for i in rs.integers(0, n_blocks, n_blocks)])
        yb = y_true[idx]
        out[b] = average_precision_score(yb, scores[idx]) if yb.sum() > 0 else np.nan
    return np.nanpercentile(out, [2.5, 50, 97.5]), out


def paired_bootstrap_diff(y_true, s_a, s_b, n_boot=1000, n_blocks=None, seed=SEED):
    """P(A > B) on the SAME resamples -- the honest way to compare two PR-AUCs
    that differ by less than their individual intervals."""
    n_blocks = n_blocks or BOOTSTRAP_BLOCKS
    n = len(y_true)
    edges = np.linspace(0, n, n_blocks + 1).astype(int)
    blocks = [np.arange(edges[i], edges[i + 1]) for i in range(n_blocks)]
    rs = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([blocks[i] for i in rs.integers(0, n_blocks, n_blocks)])
        yb = y_true[idx]
        if yb.sum() == 0:
            continue
        diffs.append(average_precision_score(yb, s_a[idx])
                     - average_precision_score(yb, s_b[idx]))
    diffs = np.array(diffs)
    return float(np.mean(diffs)), float(np.mean(diffs > 0)), \
           tuple(np.percentile(diffs, [2.5, 97.5]))


# --------------------------------------------------------------------------- #
# Canonical results store  --  INVARIANT 3 and 5
# --------------------------------------------------------------------------- #
RESULT_COLUMNS = ["run_id", "experiment", "dataset", "split_type", "fold",
                  "regime", "detector", "rho_assumed", "rho_used",
                  "metric", "value", "n_test", "n_pos_test"]

RESULTS: List[dict] = []


def _run_id(**kw) -> str:
    key = json.dumps(kw, sort_keys=True, default=str)
    return hashlib.md5(key.encode()).hexdigest()[:12]


def record(experiment: str, sp: Split, regime: str, detector: str,
           rho_assumed: float, rho_used: float, metrics: Dict[str, float],
           **extra):
    rid = _run_id(experiment=experiment, dataset=sp.dataset,
                  split_type=sp.split_type, fold=sp.fold, regime=regime,
                  detector=detector, rho=rho_assumed, **extra)
    for k, v in metrics.items():
        RESULTS.append({
            "run_id": rid, "experiment": experiment, "dataset": sp.dataset,
            "split_type": sp.split_type, "fold": sp.fold, "regime": regime,
            "detector": detector, "rho_assumed": rho_assumed,
            "rho_used": rho_used, "metric": k, "value": v,
            "n_test": len(sp.yte), "n_pos_test": int(sp.yte.sum()),
            **extra,
        })
    return rid


def results_frame() -> pd.DataFrame:
    return pd.DataFrame(RESULTS)


def pivot(experiment: str, metric: str = "PR_AUC", **filters) -> pd.DataFrame:
    df = results_frame()
    df = df[(df.experiment == experiment) & (df.metric == metric)]
    for k, v in filters.items():
        df = df[df[k] == v]
    return df


def require(*names, cell: str = ""):
    """Fail fast and legibly when a cell is run out of order."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            f"Run {cell} first -- this cell needs {', '.join(missing)}.\n"
            "Order: 0-3 setup -> 4 headline (defines SPLITS) -> 5-11 "
            "experiments -> 12 label efficiency (defines le_ulb) -> "
            "13 bootstrap (defines SCORE_CACHE) -> 14 operational -> "
            "15 tables and figures.")


def save_results(merge_with_disk: bool = True):
    """Persist the canonical frame WITHOUT destroying earlier work.

    RESULTS lives in memory. Restarting the kernel and re-running only some
    cells would otherwise overwrite a complete file with a partial one, and
    every number in the manuscript is meant to be traceable to this file.
    Rows are keyed on (run_id, metric): a re-run of the same configuration
    replaces its own rows and leaves every other experiment intact.
    """
    df = results_frame()
    p = os.path.join(OUT_DIR, "canonical_results.csv")
    if merge_with_disk and os.path.exists(p) and len(df):
        old = pd.read_csv(p)
        keep = ~old.set_index(["run_id", "metric"]).index.isin(
            df.set_index(["run_id", "metric"]).index)
        n_kept = int(keep.sum())
        df = pd.concat([old[keep], df], ignore_index=True)
        if n_kept:
            print(f"  merged with {n_kept} rows already on disk")
    df.to_csv(p, index=False)
    print(f"  canonical results -> {p}  ({len(df)} rows, "
          f"{df.run_id.nunique()} runs, "
          f"{df.experiment.nunique()} experiments)")
    return p

# =============================================================================
# 4. MAIN RUNNER  --  one split, three regimes, five detectors + ensemble
# =============================================================================

def run_split(sp: Split, experiment: str,
              regimes=("U", "N", "O"), rho_assumed: float = RHO_DEFAULT,
              hyper: Optional[dict] = None, seed: int = SEED,
              keep_scores: bool = False, verbose: bool = True):
    """Fit every detector under every regime, calibrate on validation, evaluate
    on test, and push every metric into the canonical store."""
    hyper = hyper or {}
    store = {}
    for regime in regimes:
        Xref = reference_set(sp, regime)
        rho  = rho_for(sp, regime, rho_assumed)
        cals, s_test_all, s_val_all = {}, {}, {}

        for name, fn in DETECTORS.items():
            kw = dict(hyper.get(name, {}))
            if name == "OneClassSVM":
                kw.setdefault("nu", rho)          # a-priori rho, not labels
            sc = fn(Xref, sp.Xva, sp.Xte, seed=seed, **kw)

            cal = ValidationCalibrator(sc.val)
            assert_calibration_is_clean(cal, sp)
            y_pred = cal.predict(sc.test, rho)
            m = compute_metrics(sp.yte, y_pred, sc.test)
            m["tie_ratio"] = cal.tie_ratio
            m["fit_seconds"] = sc.fit_seconds
            m["score_seconds"] = sc.score_seconds
            record(experiment, sp, regime, name, rho_assumed, rho, m)

            cals[name] = cal          # DBSCAN included: it now has a real score
            s_test_all[name] = sc.test
            s_val_all[name] = sc.val
            if keep_scores:
                store[(regime, name)] = sc

        # ----- ensemble: validation-ECDF rank average --------------------- #
        ens_test = ecdf_rank_average(cals, s_test_all)
        # The ensemble's own calibrator is built from the ensemble score of the
        # VALIDATION points. Deriving it from cals[m]._sorted instead would
        # give the uniform ramp 1/n..1 for every member, whose (1-rho) quantile
        # averaged ECDFs never reach -- yielding zero alerts.
        ens_val_scores = ecdf_rank_average(cals, s_val_all)
        ens_cal = ValidationCalibrator(ens_val_scores)
        y_pred = ens_cal.predict(ens_test, rho)
        m = compute_metrics(sp.yte, y_pred, ens_test)
        m["tie_ratio"] = ens_cal.tie_ratio
        m["fit_seconds"] = 0.0
        m["score_seconds"] = 0.0
        record(experiment, sp, regime, "Ensemble", rho_assumed, rho, m)

        # INVARIANT 1 probe, logged as a metric so it lands in the results file.
        # v2 probed the ensemble only, which is the only score that combines
        # information across detectors; the individual scores are pointwise by
        # construction. We now probe all six so the claim in Section 3.7 is
        # backed for every configuration rather than for one.            [v2.1]
        ok = single_point_invariance(ens_cal, ens_test, rho)
        record(experiment, sp, regime, "Ensemble", rho_assumed, rho,
               {"single_point_invariant": float(ok)}, check="invariance")
        for _m in s_test_all:
            _ok = single_point_invariance(cals[_m], s_test_all[_m], rho)
            record(experiment, sp, regime, _m, rho_assumed, rho,
                   {"single_point_invariant": float(_ok)}, check="invariance")
        if verbose:
            sub = pivot(experiment, "PR_AUC", dataset=sp.dataset,
                        regime=regime, fold=sp.fold)
            print(f"  regime {regime} (rho={rho:.5f}) "
                  + " | ".join(f"{r.detector}={r.value:.4f}"
                               for r in sub.itertuples()))
    return store


outputs -> C:\Users\LENOVO\Desktop\Unsupervised and Semi-Supervised Anomaly Detection for Credit Card Fraud\2_Revision_R2\resultats_R2
  ULB_PATH     OK  C:\Users\LENOVO\Desktop\Unsupervised and Semi-Supervised Anomaly Detection for Credit Card Fraud\creditcard.csv  (151 MB)
               header: "Time","V1","V2","V3","V4","V5","V6","V7","V8","V9","V10","V11","V12","V13","V14","V15","V
  PAYSIM_PATH  OK  C:\Users\LENOVO\Desktop\Unsupervised and Semi-Supervised Anomaly Detection for Credit Card Fraud\paysim.csv  (494 MB)
               header: step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceD
{
  "created": "2026-10-14T10:06:51.817048",
  "python": "3.11.0",
  "numpy": "1.26.4",
  "pandas": "2.2.2",
  "sklearn": "1.5.0",
  "scipy": "1.13.1",
  "seed": 42,
  "rho_grid": [
    0.001,
    0.005,
    0.01,
    0.02
  ],
  "rho_default": 0.005,
  "subsample_rule": "random",
  "pipeline_version": "v2.1",
  "min_core_points": 50,
  "require_nondegenera

In [3]:
# =============================================================================
# 3. R2 HELPERS  --  checkpoints, input variants, exact fast block bootstrap
# =============================================================================
import math, glob
from sklearn.metrics import average_precision_score, roc_auc_score

UNITS  = os.path.join(OUT_DIR, "units");   os.makedirs(UNITS, exist_ok=True)
TABLES = os.path.join(OUT_DIR, "tables");  os.makedirs(TABLES, exist_ok=True)
FIGS   = os.path.join(OUT_DIR, "figures"); os.makedirs(FIGS, exist_ok=True)
MEMBERS5 = list(DETECTORS)
MEMBERS4 = [m for m in MEMBERS5 if m != "DBSCAN"]
ENS4 = "Ensemble4"                 # ECDF average of the four members without DBSCAN
GRID_EVAL_CAP = 200_000            # as in R1 cell 8: PaySim grid windows (most recent rows)
_T0 = time.time()


def stamp(msg):
    line = f"[{time.strftime('%H:%M:%S')}  +{(time.time() - _T0) / 60:6.1f} min] {msg}"
    print(line, flush=True)
    with open(os.path.join(OUT_DIR, "log_r2.txt"), "a", encoding="utf-8") as f:
        f.write(line + "\n")


def is_done(unit):
    return os.path.exists(os.path.join(UNITS, unit + ".done"))


def mark_done(unit, info=None):
    with open(os.path.join(UNITS, unit + ".done"), "w") as f:
        json.dump(info or {}, f)


# --------------------------------------------------------------------------- #
# Input variants. The clock is removed AFTER the sort by step, exactly as in
# the clock-ablation cell of R1, so the chronological split is unchanged.
# --------------------------------------------------------------------------- #
_PREP_ORIG = {"ULB": _prep_ulb, "PaySim": _prep_paysim}


def _without(prep, col):
    def f(df):
        X, y, rob = prep(df)
        assert col in X.columns, f"{col} not among the features"
        return X.drop(columns=[col]), y, [c for c in rob if c != col]
    return f


def set_inputs(drop_step=False, residuals=True):
    global _prep_paysim, PAYSIM_BALANCE_FEATURES
    _prep_paysim = (_without(_PREP_ORIG["PaySim"], "step") if drop_step
                    else _PREP_ORIG["PaySim"])
    PAYSIM_BALANCE_FEATURES = residuals
    clear_raw_cache()


def reset_inputs():
    set_inputs(drop_step=False, residuals=True)


def run_unit(unit, sp, regimes, save_scores=False, experiment=None):
    """One run_split call, checkpointed. Metrics go to units/<unit>.csv and,
    if requested, validation and test scores to units/scores_<unit>.npz."""
    if is_done(unit):
        stamp(f"{unit}: already done, skipped")
        return None
    stamp(f"{unit}: start")
    sp.describe()
    RESULTS.clear()
    store = run_split(sp, experiment=experiment or unit, regimes=regimes,
                      keep_scores=save_scores, verbose=True)
    results_frame().to_csv(os.path.join(UNITS, unit + ".csv"), index=False)
    if save_scores:
        out = {"__yva": sp.yva, "__yte": sp.yte}
        for rg in regimes:
            out[f"{rg}|__rho"] = np.array(rho_for(sp, rg, RHO_DEFAULT))
            for det in MEMBERS5:
                out[f"{rg}|{det}|val"] = store[(rg, det)].val
                out[f"{rg}|{det}|test"] = store[(rg, det)].test
                for k, v in store[(rg, det)].extra.items():
                    out[f"{rg}|{det}|extra|{k}"] = np.array(v)
        np.savez(os.path.join(UNITS, f"scores_{unit}.npz"), **out)
    RESULTS.clear()
    mark_done(unit, {"n_test": int(len(sp.yte)), "n_pos_test": int(sp.yte.sum()),
                     "n_features": int(sp.Xtr.shape[1])})
    stamp(f"{unit}: done")
    return store


def unit_metrics(unit):
    p = os.path.join(UNITS, unit + ".csv")
    return pd.read_csv(p) if os.path.exists(p) else None


def ensemble_scores(z, regime, members, part):
    """ECDF average exactly as in run_split: each member mapped through the
    empirical distribution function of its own validation scores."""
    cals = {m: ValidationCalibrator(z[f"{regime}|{m}|val"]) for m in members}
    return np.mean(np.column_stack([cals[m].ecdf(z[f"{regime}|{m}|{part}"])
                                    for m in members]), axis=1)


# --------------------------------------------------------------------------- #
# Exact fast block bootstrap. block_bootstrap_ap() and paired_bootstrap_diff()
# above resample the 50 temporal blocks with rs.integers(0, 50, 50) and call
# average_precision_score on the resampled arrays. Average precision depends
# only on the multiset of (score, label) pairs, so the same value is obtained
# from block multiplicities without building the arrays. Same seed, same
# sequence of draws, same skipped replicates: the intervals of Tables 8 and 9
# are reproduced to the last digit (checked in A2).
# --------------------------------------------------------------------------- #
class FastAP:
    def __init__(self, y, s, n_blocks=BOOTSTRAP_BLOCKS):
        y = np.asarray(y).astype(np.int64); s = np.asarray(s, dtype=np.float64)
        n = len(y)
        edges = np.linspace(0, n, n_blocks + 1).astype(int)
        block = np.repeat(np.arange(n_blocks), np.diff(edges))
        order = np.argsort(s, kind="mergesort")[::-1]
        ss, ys, bs = s[order], y[order], block[order]
        ends = np.r_[np.where(np.diff(ss))[0], n - 1]
        starts = np.r_[0, ends[:-1] + 1]
        keep = np.where(np.add.reduceat(ys, starts) > 0)[0]
        cum_pos = np.zeros((len(keep), n_blocks)); cum_neg = np.zeros((len(keep), n_blocks))
        in_pos = np.zeros((len(keep), n_blocks))
        pos_c = np.zeros(n_blocks); neg_c = np.zeros(n_blocks); prev = 0
        for j, g in enumerate(keep):
            st, en = starts[g], ends[g] + 1
            if st > prev:
                pos_c += np.bincount(bs[prev:st], weights=ys[prev:st], minlength=n_blocks)
                neg_c += np.bincount(bs[prev:st], weights=1 - ys[prev:st], minlength=n_blocks)
            gp = np.bincount(bs[st:en], weights=ys[st:en], minlength=n_blocks)
            gn = np.bincount(bs[st:en], weights=1 - ys[st:en], minlength=n_blocks)
            pos_c += gp; neg_c += gn
            cum_pos[j] = pos_c; cum_neg[j] = neg_c; in_pos[j] = gp
            prev = en
        self.cum_pos, self.cum_neg, self.in_pos = cum_pos, cum_neg, in_pos
        self.total_pos = np.bincount(bs, weights=ys, minlength=n_blocks)

    def ap(self, counts):
        c = counts.astype(np.float64)
        P = float(self.total_pos @ c)
        if P <= 0:
            return np.nan
        tp = self.cum_pos @ c; fp = self.cum_neg @ c; d = self.in_pos @ c
        m = d > 0
        return float(np.sum(d[m] / P * tp[m] / (tp[m] + fp[m])))


def _block_counts(n_boot=N_BOOTSTRAP, n_blocks=BOOTSTRAP_BLOCKS, seed=SEED):
    rs = np.random.default_rng(seed)
    for _ in range(n_boot):
        yield np.bincount(rs.integers(0, n_blocks, n_blocks), minlength=n_blocks)


def boot_ci(y, s):
    """= block_bootstrap_ap(y, s): 2.5, 50 and 97.5 percentiles."""
    f = FastAP(y, s)
    out = np.array([f.ap(c) for c in _block_counts()])
    return np.nanpercentile(out, [2.5, 50, 97.5])


def boot_paired(y, s_a, s_b):
    """= paired_bootstrap_diff(y, s_a, s_b, n_boot=2000), plus a two-sided
    bootstrap p-value, p = 2 (min(#{d<=0}, #{d>=0}) + 1) / (B + 1)."""
    fa, fb = FastAP(y, s_a), FastAP(y, s_b)
    d = []
    for c in _block_counts():
        a = fa.ap(c)
        if np.isnan(a):
            continue
        d.append(a - fb.ap(c))
    d = np.array(d)
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = min(1.0, 2.0 * (min(np.sum(d <= 0), np.sum(d >= 0)) + 1) / (len(d) + 1))
    return {"diff": float(d.mean()), "P_a_ahead": float(np.mean(d > 0)),
            "ci_lo": float(lo), "ci_hi": float(hi), "p_boot": float(p), "B": len(d)}


def holm(p):
    """Holm step-down adjusted p-values (family-wise error rate)."""
    p = np.asarray(p, dtype=float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); running = 0.0
    for k, i in enumerate(order):
        running = max(running, (m - k) * p[i]); adj[i] = min(1.0, running)
    return adj


print("helpers ready; units ->", UNITS)


helpers ready; units -> C:\Users\LENOVO\Desktop\Unsupervised and Semi-Supervised Anomaly Detection for Credit Card Fraud\2_Revision_R2\resultats_R2\units


In [4]:
# =============================================================================
# S1. ENVIRONMENT AND REPRODUCTION CHECK ON ULB  (about 2 min)
#     Re-runs the R1 headline configuration (regimes U, N, O) and compares it
#     with canonical_results.csv of the R1 run. Saves validation and test
#     scores, which every later analysis reads instead of refitting.
# =============================================================================
import sklearn, scipy, platform
_r1m = json.load(open(os.path.join(R1_DIR, "manifest.json")))
_env = {"python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "sklearn": sklearn.__version__,
        "scipy": scipy.__version__}
print("Environment (R1 run in brackets):")
for _k, _v in _env.items():
    _flag = "" if str(_r1m.get(_k)) == _v else "   <-- differs from the R1 run"
    print(f"  {_k:<8} {_v:<10} ({_r1m.get(_k)}){_flag}")

CANON_R1 = pd.read_csv(os.path.join(R1_DIR, "canonical_results.csv"))
HEAD_R1 = CANON_R1[(CANON_R1.experiment == "headline")
                   & (CANON_R1.split_type == "chronological") & (CANON_R1.fold == 0)]
REPRO = {}


def check_headline(unit, ds):
    new = unit_metrics(unit)
    m = new.merge(HEAD_R1[HEAD_R1.dataset == ds],
                  on=["dataset", "regime", "detector", "metric"],
                  suffixes=("_new", "_r1"))
    m = m[m.metric.isin(["PR_AUC", "ROC_AUC", "TP", "FP", "n_alerts"])]
    m = m.assign(absdiff=(m.value_new.astype(float) - m.value_r1.astype(float)).abs())
    expected = 3 * 6 * 5
    worst = float(m.absdiff.max()) if len(m) else np.inf
    ok = (len(m) == expected) and worst < 1e-6
    REPRO[ds] = {"rows_compared": int(len(m)), "rows_expected": expected,
                 "max_abs_diff": worst, "reproduced": bool(ok)}
    print(f"  {ds}: {len(m)}/{expected} values compared, largest |difference| "
          f"= {worst:.2e}  ->  {'REPRODUCED' if ok else 'NOT REPRODUCED'}")
    if not ok:
        print(m.sort_values("absdiff", ascending=False).head(10)[
            ["regime", "detector", "metric", "value_new", "value_r1", "absdiff"]]
            .to_string(index=False))
        if STOP_IF_NOT_REPRODUCED:
            raise RuntimeError(
                f"{ds}: the R1 headline values are not reproduced in this "
                "environment. Run the notebook in the Python environment of the "
                "R1 run (versions above), then delete the folder "
                f"{UNITS} and run again.")
    # test scores against the scores saved by R1, for information
    z_old = np.load(os.path.join(R1_DIR, "test_scores.npz"))
    z_new = np.load(os.path.join(UNITS, f"scores_{unit}.npz"))
    if z_new["__yte"].shape == z_old[f"{ds}|U|__y"].shape:
        worst_s = max(float(np.max(np.abs(z_new[f"{rg}|{d}|test"].astype(float)
                                          - z_old[f"{ds}|{rg}|{d}"].astype(float))))
                      for rg in ("U", "N") for d in MEMBERS5)
        print(f"  {ds}: largest |test score difference| vs R1 test_scores.npz = {worst_s:.2e}")
        REPRO[ds]["max_abs_score_diff"] = worst_s
    else:
        print(f"  {ds}: test block differs in size from R1 test_scores.npz; scores not compared")
    with open(os.path.join(OUT_DIR, "reproduction_check.json"), "w") as f:
        json.dump({"environment": _env, "r1_manifest": _r1m, "checks": REPRO}, f, indent=2)


reset_inputs()
if not is_done("headline_ULB"):
    run_unit("headline_ULB", chronological_split("ULB"), ("U", "N", "O"),
             save_scores=True, experiment="headline")
check_headline("headline_ULB", "ULB")


Environment (R1 run in brackets):
  python   3.11.0     (3.11.0)
  numpy    1.26.4     (1.26.4)
  pandas   2.2.2      (2.2.2)
  sklearn  1.5.0      (1.5.0)
  scipy    1.13.1     (1.13.1)
  raw cache cleared
  ULB: 90/90 values compared, largest |difference| = 0.00e+00  ->  REPRODUCED
  ULB: largest |test score difference| vs R1 test_scores.npz = 1.14e-05


In [5]:
# =============================================================================
# S2. PaySim HEADLINE RE-RUN AND REPRODUCTION CHECK  (about 35-45 min)
#     Same configuration as Table 12 (step and the balance residuals as inputs).
# =============================================================================
reset_inputs()
if not is_done("headline_PaySim"):
    _sp = chronological_split("PaySim")
    run_unit("headline_PaySim", _sp, ("U", "N", "O"), save_scores=True,
             experiment="headline")
    del _sp
check_headline("headline_PaySim", "PaySim")


  raw cache cleared
  PaySim: 90/90 values compared, largest |difference| = 3.10e-09  ->  REPRODUCED
  PaySim: largest |test score difference| vs R1 test_scores.npz = 3.81e-06


In [6]:
# =============================================================================
# S3. PaySim INPUT ABLATION (Table 13), scores kept   (about 60-90 min)
#     residuals removed / step removed / both removed; regimes U and N.
#     Each run is checked against the R1 Table 13 values (4 decimals).
# =============================================================================
R1_TABLE13 = {   # (regime, detector): (clock and residuals, residuals removed, clock removed, both removed)
    ("N", "DBSCAN"): (0.0248, 0.0226, 0.0215, 0.0192),
    ("N", "Ensemble"): (0.1706, 0.0420, 0.0686, 0.0506),
    ("N", "IsolationForest"): (0.0165, 0.0137, 0.0124, 0.0151),
    ("N", "KMeans"): (0.0458, 0.0369, 0.0381, 0.0311),
    ("N", "LOF"): (0.0030, 0.0025, 0.1905, 0.0717),
    ("N", "OneClassSVM"): (0.1018, 0.0470, 0.1048, 0.0529),
    ("U", "DBSCAN"): (0.0248, 0.0556, 0.0215, 0.0192),
    ("U", "Ensemble"): (0.1502, 0.0191, 0.0590, 0.0516),
    ("U", "IsolationForest"): (0.0161, 0.0143, 0.0129, 0.0159),
    ("U", "KMeans"): (0.0419, 0.0369, 0.0366, 0.0311),
    ("U", "LOF"): (0.0031, 0.0026, 0.1483, 0.0684),
    ("U", "OneClassSVM"): (0.1054, 0.0552, 0.0849, 0.0506),
}
ABLATIONS = [("PaySim_noresid", False, False, 1),   # unit, drop step, residuals, column of R1_TABLE13
             ("PaySim_nostep",  True,  True,  2),
             ("PaySim_noboth",  True,  False, 3)]
T13_CHECK = []
for _unit, _drop, _res, _col in ABLATIONS:
    if not is_done(_unit):
        set_inputs(drop_step=_drop, residuals=_res)
        try:
            _sp = chronological_split("PaySim")
            run_unit(_unit, _sp, ("U", "N"), save_scores=True, experiment=_unit)
            del _sp
        finally:
            reset_inputs()
    _m = unit_metrics(_unit)
    _m = _m[_m.metric == "PR_AUC"]
    for _r in _m.itertuples():
        _ref = R1_TABLE13[(_r.regime, _r.detector)][_col]
        T13_CHECK.append({"unit": _unit, "regime": _r.regime, "detector": _r.detector,
                          "PR_AUC": _r.value, "R1_table13": _ref,
                          "matches_4dp": bool(abs(_r.value - _ref) <= 0.0000501)})
T13_CHECK = pd.DataFrame(T13_CHECK)
T13_CHECK.to_csv(os.path.join(TABLES, "check_table13_vs_R1.csv"), index=False)
print(T13_CHECK.to_string(index=False))
_bad = T13_CHECK[~T13_CHECK.matches_4dp]
if len(_bad):
    print(f"\n!! {len(_bad)} Table 13 value(s) differ from R1 at 4 decimals (listed above).")
    if STOP_IF_NOT_REPRODUCED:
        raise RuntimeError("Table 13 of R1 is not reproduced: stop and send the log.")
else:
    print("\nTable 13 of R1 reproduced at 4 decimals for every cell.")


          unit regime        detector   PR_AUC  R1_table13  matches_4dp
PaySim_noresid      U IsolationForest 0.014251      0.0143         True
PaySim_noresid      U             LOF 0.002559      0.0026         True
PaySim_noresid      U     OneClassSVM 0.055152      0.0552         True
PaySim_noresid      U          DBSCAN 0.055634      0.0556         True
PaySim_noresid      U          KMeans 0.036851      0.0369         True
PaySim_noresid      U        Ensemble 0.019051      0.0191         True
PaySim_noresid      N IsolationForest 0.013739      0.0137         True
PaySim_noresid      N             LOF 0.002526      0.0025         True
PaySim_noresid      N     OneClassSVM 0.046979      0.0470         True
PaySim_noresid      N          DBSCAN 0.022621      0.0226         True
PaySim_noresid      N          KMeans 0.036850      0.0369         True
PaySim_noresid      N        Ensemble 0.042027      0.0420         True
 PaySim_nostep      U IsolationForest 0.012886      0.0129      

In [7]:
# =============================================================================
# S4. PaySim ROLLING ORIGINS WITHOUT step   (about 60-90 min)      [R1 point 3]
#     Same five expanding-window origins as Table 11 (rolling_origin_splits),
#     with the clock removed as an input; residuals kept, as in Table 12.
# =============================================================================
_todo = [k for k in range(FOLD_CONFIG["PaySim"][0])
         if not is_done(f"PaySim_rolling_nostep_fold{k}")]
if _todo:
    set_inputs(drop_step=True, residuals=True)
    try:
        for _sp in rolling_origin_splits("PaySim"):
            if _sp.fold in _todo:
                run_unit(f"PaySim_rolling_nostep_fold{_sp.fold}", _sp, ("U", "N"),
                         experiment="rolling_origin_nostep")
            del _sp
    finally:
        reset_inputs()
else:
    stamp("S4: all five origins already done")


[10:06:54  +   0.0 min] S4: all five origins already done


In [8]:
# =============================================================================
# S5. PaySim RANDOM AND PREVALENCE-MATCHED ARMS WITHOUT step  (about 80-110 min)
#     Same arms as Table 10 (random_split, three random_prevalence_matched draws
#     with seeds 42-44), with the clock removed. The chronological arm without
#     step is unit PaySim_nostep of S3.                              [R1 point 3]
# =============================================================================
set_inputs(drop_step=True, residuals=True)
try:
    _ref = json.load(open(os.path.join(UNITS, "PaySim_nostep.done")))
    _n_te, _n_pos = _ref["n_test"], _ref["n_pos_test"]
    if not is_done("PaySim_random_nostep"):
        run_unit("PaySim_random_nostep", random_split("PaySim"), ("U", "N"),
                 experiment="split_comparison_nostep")
    for _s in range(3):
        _u = f"PaySim_matched_nostep_draw{_s}"
        if not is_done(_u):
            _sp = random_prevalence_matched("PaySim", _n_te, _n_pos, seed=SEED + _s)
            _sp.fold = _s
            run_unit(_u, _sp, ("U", "N"), experiment="split_comparison_nostep")
            del _sp
finally:
    reset_inputs()


  raw cache cleared
  raw cache cleared


In [9]:
# =============================================================================
# S6. ENSEMBLE REBUILT FROM THE VALIDATION-SELECTED MEMBERS  (about 25-35 min)
#     Notebook cell 8b of R1 (never executed there).                [R1 point 4]
#     Members are fitted on the full reference set at the configuration
#     selected in Table 18; each member's ECDF is estimated on the full
#     validation window, as for the headline ensemble (Section 3.5). On ULB
#     the test window is the full test block. On PaySim, as in Table 18, test
#     PR-AUC is computed on the most recent 200,000 test rows (GRID_EVAL_CAP);
#     the fixed-configuration ensemble scored this way is the R1 reference of
#     Section 4.9 (0.2426 under U, 0.2403 under N), which is checked below.
#     Validation PR-AUC is reported on the window used for selection.
# =============================================================================
SELECTED_R1 = {   # (dataset, regime, detector): configuration reported in Table 18
    ("ULB", "N", "IsolationForest"): {"n_estimators": 400},
    ("ULB", "N", "LOF"): {"n_neighbors": 20},
    ("ULB", "N", "OneClassSVM"): {"gamma": "scale"},
    ("ULB", "N", "DBSCAN"): {"eps_scale": 4.0},
    ("ULB", "N", "KMeans"): {"k": 32},
    ("ULB", "U", "IsolationForest"): {"n_estimators": 100},
    ("ULB", "U", "LOF"): {"n_neighbors": 50},
    ("ULB", "U", "OneClassSVM"): {"gamma": "scale"},
    ("ULB", "U", "DBSCAN"): {"min_samples": 5},
    ("ULB", "U", "KMeans"): {"k": 16},
    ("PaySim", "N", "IsolationForest"): {"n_estimators": 200},
    ("PaySim", "N", "LOF"): {"n_neighbors": 5},
    ("PaySim", "N", "OneClassSVM"): {"gamma": 0.1},
    ("PaySim", "N", "DBSCAN"): {"min_samples": 25},
    ("PaySim", "N", "KMeans"): {"k": 32},
    ("PaySim", "U", "IsolationForest"): {"n_estimators": 100},   # 100 and 200 tie on validation
    ("PaySim", "U", "LOF"): {"n_neighbors": 5},
    ("PaySim", "U", "OneClassSVM"): {"gamma": 0.1},
    ("PaySim", "U", "DBSCAN"): {"eps_scale": 0.5},
    ("PaySim", "U", "KMeans"): {"k": 32},
}
R1_FIXED_ENSEMBLE_ON_WINDOW = {("PaySim", "U"): 0.2426, ("PaySim", "N"): 0.2403}


def tuned_ensemble(ds):
    unit = f"tuned_ensemble_{ds}"
    if is_done(unit):
        stamp(f"{unit}: already done, skipped"); return
    stamp(f"{unit}: start")
    reset_inputs()
    sp = chronological_split(ds)
    if len(sp.Xte) > GRID_EVAL_CAP:
        wv, wt = slice(-GRID_EVAL_CAP, None), slice(-GRID_EVAL_CAP, None)
        window = f"most recent {GRID_EVAL_CAP:,} validation and test rows"
    else:
        wv, wt = slice(None), slice(None)
        window = "full validation and test windows"
    Xte_w, yte_w, yva_w = sp.Xte[wt], sp.yte[wt], sp.yva[wv]
    rows = []
    for regime in ("U", "N"):
        Xref = reference_set(sp, regime)
        rho = rho_for(sp, regime, RHO_DEFAULT)
        for setting in ("fixed", "selected"):
            sc = {}
            for det in MEMBERS5:
                cfg = SELECTED_R1[(ds, regime, det)] if setting == "selected" else {}
                kw = dict(cfg)
                if det == "OneClassSVM":
                    kw.setdefault("nu", rho)
                sc[det] = DETECTORS[det](Xref, sp.Xva, Xte_w, seed=SEED, **kw)
                rows.append({"dataset": ds, "regime": regime, "setting": setting,
                             "detector": det, "config": json.dumps(cfg),
                             "val_PR_AUC": average_precision_score(yva_w, sc[det].val[wv]),
                             "test_PR_AUC": average_precision_score(yte_w, sc[det].test),
                             "window": window})
            cals = {m: ValidationCalibrator(sc[m].val) for m in MEMBERS5}   # full validation window
            for name, mem in (("Ensemble", MEMBERS5), (ENS4, MEMBERS4)):
                ev = ecdf_rank_average({m: cals[m] for m in mem}, {m: sc[m].val for m in mem})
                et = ecdf_rank_average({m: cals[m] for m in mem}, {m: sc[m].test for m in mem})
                rows.append({"dataset": ds, "regime": regime, "setting": setting,
                             "detector": name, "config": f"members at the {setting} configuration",
                             "val_PR_AUC": average_precision_score(yva_w, ev[wv]),
                             "test_PR_AUC": average_precision_score(yte_w, et),
                             "window": window})
            stamp(f"  {ds}/{regime}/{setting}: " + " | ".join(
                f"{r['detector']} {r['test_PR_AUC']:.4f}" for r in rows[-7:]))
            if setting == "fixed" and (ds, regime) in R1_FIXED_ENSEMBLE_ON_WINDOW:
                v = [r["test_PR_AUC"] for r in rows if r["dataset"] == ds and r["regime"] == regime
                     and r["setting"] == "fixed" and r["detector"] == "Ensemble"][0]
                ref = R1_FIXED_ENSEMBLE_ON_WINDOW[(ds, regime)]
                print(f"    fixed-configuration ensemble on the window: {v:.4f} (R1: {ref})"
                      + ("" if abs(v - ref) <= 0.0000501 else "   <-- differs from R1"))
    pd.DataFrame(rows).to_csv(os.path.join(UNITS, unit + ".csv"), index=False)
    mark_done(unit)
    stamp(f"{unit}: done")


for _ds in ["ULB", "PaySim"]:
    tuned_ensemble(_ds)


[10:59:59  +  53.1 min] tuned_ensemble_ULB: already done, skipped
[10:59:59  +  53.1 min] tuned_ensemble_PaySim: already done, skipped


In [10]:
# =============================================================================
# S7. FIVE REFERENCE-SET DRAWS, ENSEMBLES INCLUDED  (ULB 3 min; PaySim 75-100 min)
#     As the subsampling study of R1 (Table 19): LOF, One-Class SVM and DBSCAN
#     refitted with seeds 42-46 (seed 42 = the reference draw of Tables 4 and
#     12); Isolation Forest and K-Means, which use no capped subsample, stay at
#     seed 42. The five- and four-member ECDF ensembles are rebuilt per draw.
#                                                              [R1 points 1, 6]
# =============================================================================
DRAW_SEEDS = [SEED + s for s in range(N_SUBSAMPLE_SEEDS)]


def five_draws(ds, tag="ref", drop_step=False, residuals=True):
    for regime in ("U", "N"):
        unit = f"draws_{ds}_{tag}_{regime}"
        if is_done(unit):
            stamp(f"{unit}: already done, skipped"); continue
        stamp(f"{unit}: start")
        set_inputs(drop_step=drop_step, residuals=residuals)
        try:
            sp = chronological_split(ds)
            Xref = reference_set(sp, regime)
            rho = rho_for(sp, regime, RHO_DEFAULT)
            fixed = {d: DETECTORS[d](Xref, sp.Xva, sp.Xte, seed=SEED)
                     for d in ("IsolationForest", "KMeans")}
            rows = []
            for s, seed in enumerate(DRAW_SEEDS):
                sc = dict(fixed)
                for d in ("LOF", "OneClassSVM", "DBSCAN"):
                    kw = {"nu": rho} if d == "OneClassSVM" else {}
                    sc[d] = DETECTORS[d](Xref, sp.Xva, sp.Xte, seed=seed, **kw)
                cals = {m: ValidationCalibrator(sc[m].val) for m in MEMBERS5}
                te = {m: sc[m].test for m in MEMBERS5}
                te["Ensemble"] = ecdf_rank_average(cals, {m: sc[m].test for m in MEMBERS5})
                te[ENS4] = ecdf_rank_average({m: cals[m] for m in MEMBERS4},
                                             {m: sc[m].test for m in MEMBERS4})
                for name, s_te in te.items():
                    rows.append({"dataset": ds, "inputs": tag, "regime": regime,
                                 "draw": s, "seed": seed, "detector": name,
                                 "PR_AUC": average_precision_score(sp.yte, s_te),
                                 "ROC_AUC": roc_auc_score(sp.yte, s_te),
                                 "n_core": (sc["DBSCAN"].extra.get("n_core")
                                            if name == "DBSCAN" else np.nan)})
                stamp(f"  {ds}/{tag}/{regime} seed {seed}: " + " | ".join(
                    f"{r['detector']} {r['PR_AUC']:.4f}" for r in rows[-7:]))
            pd.DataFrame(rows).to_csv(os.path.join(UNITS, unit + ".csv"), index=False)
            mark_done(unit)
            del sp, Xref, fixed
        finally:
            reset_inputs()
        stamp(f"{unit}: done")


five_draws("ULB")
five_draws("PaySim")
if RUN_OPTIONAL:
    five_draws("PaySim", "nostep", drop_step=True, residuals=True)
    five_draws("PaySim", "noboth", drop_step=True, residuals=False)


[10:59:59  +  53.1 min] draws_ULB_ref_U: already done, skipped
[10:59:59  +  53.1 min] draws_ULB_ref_N: already done, skipped
[10:59:59  +  53.1 min] draws_PaySim_ref_U: already done, skipped
[10:59:59  +  53.1 min] draws_PaySim_ref_N: already done, skipped


In [11]:
# =============================================================================
# A1. FOUR-MEMBER ENSEMBLE (without DBSCAN) BESIDE THE FIVE-MEMBER ONE,
#     ALERT BUDGETS AS ceil(rho * N), AGGREGATION SCHEMES     [R1 points 6, 9d]
#     From the saved scores only: nothing is refitted.
# =============================================================================
from sklearn.linear_model import LogisticRegression


def load_scores(unit):
    p = os.path.join(UNITS, f"scores_{unit}.npz")
    return np.load(p) if os.path.exists(p) else None


# ---- Tables 4 and 12: threshold metrics of both ensembles ------------------- #
ens_rows = []
for ds in ["ULB", "PaySim"]:
    z = load_scores(f"headline_{ds}")
    if z is None:
        print(f"  {ds}: headline scores missing, skipped"); continue
    yte = z["__yte"]
    rec = unit_metrics(f"headline_{ds}")
    for rg in ["U", "N", "O"]:
        rho = float(z[f"{rg}|__rho"])
        for name, mem in (("Ensemble", MEMBERS5), (ENS4, MEMBERS4)):
            ev, et = ensemble_scores(z, rg, mem, "val"), ensemble_scores(z, rg, mem, "test")
            m = compute_metrics(yte, ValidationCalibrator(ev).predict(et, rho), et)
            if name == "Ensemble":      # must equal what run_split recorded
                r = rec[(rec.regime == rg) & (rec.detector == "Ensemble")
                        & (rec.metric == "PR_AUC")].value.iloc[0]
                assert abs(r - m["PR_AUC"]) < 1e-12, (ds, rg, r, m["PR_AUC"])
            ens_rows.append({"dataset": ds, "regime": rg, "detector": name,
                             "rho": rho, **m})
ENS_METRICS = pd.DataFrame(ens_rows)
ENS_METRICS.to_csv(os.path.join(TABLES, "A1_ensembles_5_and_4_members.csv"), index=False)
print(ENS_METRICS[["dataset", "regime", "detector", "Precision", "Recall", "F1",
                   "ROC_AUC", "PR_AUC", "TP", "FP", "n_alerts"]].round(4).to_string(index=False))

# ---- Table 5 with one convention: budget = ceil(rho * N_test) --------------- #
bud_rows = []
for ds in ["ULB", "PaySim"]:
    rec = unit_metrics(f"headline_{ds}")
    if rec is None:
        continue
    al = rec[rec.metric == "n_alerts"]
    for r in al.itertuples():
        bud_rows.append({"dataset": ds, "regime": r.regime, "detector": r.detector,
                         "rho": r.rho_used, "n_test": r.n_test,
                         "budget_ceil": int(math.ceil(r.rho_used * r.n_test - 1e-9)),
                         "alerts": int(r.value)})
    e4 = ENS_METRICS[(ENS_METRICS.dataset == ds) & (ENS_METRICS.detector == ENS4)]
    for r in e4.itertuples():
        n_te = int(rec.n_test.iloc[0])
        bud_rows.append({"dataset": ds, "regime": r.regime, "detector": ENS4,
                         "rho": r.rho, "n_test": n_te,
                         "budget_ceil": int(math.ceil(r.rho * n_te - 1e-9)),
                         "alerts": int(r.n_alerts)})
BUDGET = pd.DataFrame(bud_rows)
BUDGET["ratio"] = BUDGET.alerts / BUDGET.budget_ceil
BUDGET.to_csv(os.path.join(TABLES, "A1_alert_budget_ceil.csv"), index=False)
print("\nAlert budgets, ceil(rho N):")
print(BUDGET.round(4).to_string(index=False))

# ---- Table 7: aggregation schemes, five and four members -------------------- #
def aggregation_from_scores(z, rg, members):
    yva, yte = z["__yva"], z["__yte"]
    v = {m: z[f"{rg}|{m}|val"] for m in members}
    t = {m: z[f"{rg}|{m}|test"] for m in members}
    cals = {m: ValidationCalibrator(v[m]) for m in members}
    Ev = np.column_stack([cals[m].ecdf(v[m]) for m in members])
    Et = np.column_stack([cals[m].ecdf(t[m]) for m in members])
    mu = np.array([v[m].mean() for m in members])
    sd = np.array([v[m].std() + 1e-12 for m in members])
    Zt = (np.column_stack([t[m] for m in members]) - mu) / sd
    w = np.array([average_precision_score(yva, v[m]) for m in members]); w = w / w.sum()
    combos = {"ecdf_mean": Et.mean(axis=1), "z_mean": Zt.mean(axis=1),
              "ap_weighted": Et @ w, "ecdf_max": Et.max(axis=1)}
    clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Ev, yva)
    combos["stacking_SUPERVISED"] = clf.predict_proba(Et)[:, 1]
    return {k: average_precision_score(yte, s) for k, s in combos.items()}


agg_rows = []
for ds in ["ULB", "PaySim"]:
    z = load_scores(f"headline_{ds}")
    if z is None:
        continue
    for rg in ["U", "N"]:
        for label, mem in (("five members", MEMBERS5), ("four members, no DBSCAN", MEMBERS4)):
            agg_rows.append({"dataset": ds, "regime": rg, "members": label,
                             **aggregation_from_scores(z, rg, mem)})
AGG = pd.DataFrame(agg_rows)
AGG.to_csv(os.path.join(TABLES, "A1_aggregation_schemes_5_and_4.csv"), index=False)
_r1agg = pd.read_csv(os.path.join(R1_DIR, "table_aggregation_schemes.csv"))
_chk = []
for r in AGG[AGG.members == "five members"].itertuples():
    for sch in ["ecdf_mean", "z_mean", "ap_weighted", "ecdf_max", "stacking_SUPERVISED"]:
        old = _r1agg[(_r1agg.dataset == r.dataset) & (_r1agg.regime == r.regime)
                     & (_r1agg.scheme == sch)].PR_AUC
        if len(old):
            _chk.append(abs(getattr(r, sch) - float(old.iloc[0])))
print("\nAggregation schemes (PR-AUC):")
print(AGG.round(4).to_string(index=False))
print(f"  five-member rows vs R1 Table 7: largest |difference| = {max(_chk):.1e} "
      "(R1 file is rounded to 4 decimals)")


dataset regime  detector  Precision  Recall     F1  ROC_AUC  PR_AUC  TP   FP  n_alerts
    ULB      U  Ensemble     0.1117  0.2800 0.1597   0.9585  0.0695  21  167       188
    ULB      U Ensemble4     0.1159  0.3200 0.1702   0.9575  0.0749  24  183       207
    ULB      N  Ensemble     0.2176  0.5600 0.3134   0.9639  0.1400  42  151       193
    ULB      N Ensemble4     0.2396  0.6133 0.3446   0.9638  0.1626  46  146       192
    ULB      O  Ensemble     0.0000  0.0000 0.0000   0.9585  0.0695   0   73        73
    ULB      O Ensemble4     0.0400  0.0400 0.0400   0.9575  0.0749   3   72        75
 PaySim      U  Ensemble     0.2456  0.0557 0.0908   0.9178  0.1502 237  728       965
 PaySim      U Ensemble4     0.0967  0.0851 0.0905   0.9055  0.0798 362 3383      3745
 PaySim      N  Ensemble     0.3147  0.0978 0.1492   0.9163  0.1706 416  906      1322
 PaySim      N Ensemble4     0.1433  0.1921 0.1641   0.9053  0.0874 817 4884      5701
 PaySim      O  Ensemble     0.0417  0.0007

In [12]:
# =============================================================================
# A2. BLOCK-BOOTSTRAP INTERVALS AND PAIRED COMPARISONS       [R1 points 2, 6, 7]
#     50 temporal blocks, 2,000 replicates, seed 42: the procedure of Tables 8
#     and 9, computed exactly (FastAP) and checked against the R1 values.
# =============================================================================
def scores_with_ensembles(unit, rg):
    z = load_scores(unit)
    if z is None:
        return None, None
    s = {m: z[f"{rg}|{m}|test"] for m in MEMBERS5}
    s["Ensemble"] = ensemble_scores(z, rg, MEMBERS5, "test")
    s[ENS4] = ensemble_scores(z, rg, MEMBERS4, "test")
    return z["__yte"], s


# ---- Table 8 (+ four-member ensemble) and Table 13 intervals --------------- #
SETTINGS = [("ULB", "headline_ULB", "Time and all inputs (Table 4)"),
            ("PaySim", "headline_PaySim", "step and residuals (Table 12)"),
            ("PaySim", "PaySim_noresid", "residuals removed"),
            ("PaySim", "PaySim_nostep", "step removed"),
            ("PaySim", "PaySim_noboth", "step and residuals removed")]
ci_rows = []
for ds, unit, label in SETTINGS:
    for rg in ["U", "N"]:
        y, s = scores_with_ensembles(unit, rg)
        if y is None:
            print(f"  {unit}: scores missing, skipped"); continue
        for name, sc in s.items():
            lo, med, hi = boot_ci(y, sc)
            ci_rows.append({"dataset": ds, "inputs": label, "unit": unit, "regime": rg,
                            "detector": name, "PR_AUC": average_precision_score(y, sc),
                            "ci_lo": lo, "ci_hi": hi, "ci_width": hi - lo})
        stamp(f"  intervals done: {unit}/{rg}")
CI = pd.DataFrame(ci_rows)
CI.to_csv(os.path.join(TABLES, "A2_bootstrap_intervals.csv"), index=False)
# check against R1 Table 8
_r1ci = pd.read_csv(os.path.join(R1_DIR, "table_bootstrap_ci.csv"))
_m = CI[CI.unit.str.startswith("headline")].merge(_r1ci, on=["dataset", "regime", "detector"],
                                                 suffixes=("", "_r1"))
if len(_m):
    _d = max((_m.ci_lo - _m.ci_lo_r1).abs().max(), (_m.ci_hi - _m.ci_hi_r1).abs().max())
    print(f"Table 8 of R1 vs this run: {len(_m)} intervals, largest |difference| = {_d:.1e} "
          "(R1 file rounded to 4 decimals)")

# ---- Table 9: each ensemble against each member, Holm-adjusted -------------- #
pair_rows = []
for ds, unit in [("ULB", "headline_ULB"), ("PaySim", "headline_PaySim")]:
    for rg in ["U", "N"]:
        y, s = scores_with_ensembles(unit, rg)
        if y is None:
            continue
        best = max(MEMBERS5, key=lambda m: average_precision_score(y, s[m]))
        for ens in ("Ensemble", ENS4):
            for rival in MEMBERS5:
                r = boot_paired(y, s[ens], s[rival])
                pair_rows.append({"dataset": ds, "regime": rg, "ensemble": ens,
                                  "rival": rival, "is_best_single": rival == best,
                                  "PR_AUC_ensemble": average_precision_score(y, s[ens]),
                                  "PR_AUC_rival": average_precision_score(y, s[rival]), **r})
        stamp(f"  paired comparisons done: {unit}/{rg}")
PAIRS = pd.DataFrame(pair_rows)
if len(PAIRS):
    for ens in ("Ensemble", ENS4):
        k = PAIRS.ensemble == ens
        PAIRS.loc[k, "p_holm_within_ensemble"] = holm(PAIRS.loc[k, "p_boot"])
    PAIRS["p_holm_all"] = holm(PAIRS.p_boot)
    PAIRS["verdict_unadjusted"] = np.where(PAIRS.ci_lo > 0, "ensemble ahead",
                                  np.where(PAIRS.ci_hi < 0, "ensemble behind", "within noise"))
    PAIRS["verdict_holm"] = np.where(PAIRS.p_holm_within_ensemble < 0.05,
                                     np.where(PAIRS["diff"] > 0, "ensemble ahead", "ensemble behind"),
                                     "not resolved")
    PAIRS.to_csv(os.path.join(TABLES, "A2_paired_ensemble_vs_members.csv"), index=False)
    _r1p = pd.read_csv(os.path.join(R1_DIR, "phase0", "table_paired_bootstrap_ensemble.csv"))
    _m = PAIRS[PAIRS.ensemble == "Ensemble"].merge(_r1p, on=["dataset", "regime", "rival"],
                                                   suffixes=("", "_r1"))
    print(f"Table 9 of R1 vs this run: {len(_m)} comparisons, largest |difference| in the "
          f"mean difference = {(_m['diff'] - _m['diff_r1']).abs().max():.1e}, in the bounds = "
          f"{max((_m.ci_lo - _m.ci_lo_r1).abs().max(), (_m.ci_hi - _m.ci_hi_r1).abs().max()):.1e}")
    print(PAIRS[["dataset", "regime", "ensemble", "rival", "diff", "ci_lo", "ci_hi", "p_boot",
                 "p_holm_within_ensemble", "verdict_unadjusted", "verdict_holm"]]
          .round(4).to_string(index=False))

# ---- the PaySim ranking claims of the abstract (Table 13 settings) --------- #
claim_rows = []
for unit, label in [("PaySim_nostep", "step removed"), ("PaySim_noboth", "step and residuals removed")]:
    for rg in ["U", "N"]:
        y, s = scores_with_ensembles(unit, rg)
        if y is None:
            continue
        for a, b in [("LOF", "OneClassSVM"), ("LOF", "Ensemble"), ("LOF", ENS4),
                     ("OneClassSVM", "Ensemble")]:
            r = boot_paired(y, s[a], s[b])
            claim_rows.append({"inputs": label, "regime": rg, "a": a, "b": b,
                               "PR_AUC_a": average_precision_score(y, s[a]),
                               "PR_AUC_b": average_precision_score(y, s[b]), **r})
CLAIMS = pd.DataFrame(claim_rows)
if len(CLAIMS):
    CLAIMS["p_holm"] = holm(CLAIMS.p_boot)
    CLAIMS.to_csv(os.path.join(TABLES, "A2_paired_paysim_ablation_claims.csv"), index=False)
    print("\nPaySim without the clock: paired differences (a - b):")
    print(CLAIMS.round(4).to_string(index=False))


[11:22:30  +  75.6 min]   intervals done: headline_ULB/U
[11:22:32  +  75.7 min]   intervals done: headline_ULB/N
[11:23:06  +  76.2 min]   intervals done: headline_PaySim/U
[11:23:43  +  76.9 min]   intervals done: headline_PaySim/N
[11:24:20  +  77.5 min]   intervals done: PaySim_noresid/U
[11:24:58  +  78.1 min]   intervals done: PaySim_noresid/N
[11:25:30  +  78.6 min]   intervals done: PaySim_nostep/U
[11:26:11  +  79.3 min]   intervals done: PaySim_nostep/N
[11:26:54  +  80.0 min]   intervals done: PaySim_noboth/U
[11:27:39  +  80.8 min]   intervals done: PaySim_noboth/N
Table 8 of R1 vs this run: 24 intervals, largest |difference| = 4.9e-05 (R1 file rounded to 4 decimals)
[11:27:43  +  80.8 min]   paired comparisons done: headline_ULB/U
[11:27:47  +  80.9 min]   paired comparisons done: headline_ULB/N
[11:29:36  +  82.7 min]   paired comparisons done: headline_PaySim/U
[11:31:08  +  84.3 min]   paired comparisons done: headline_PaySim/N
Table 9 of R1 vs this run: 20 comparisons,

In [13]:
# =============================================================================
# A3. FIVE-DRAW SUMMARIES (mean, SD, range, status of the reference draw)
#     and A4. ENSEMBLE AT THE VALIDATION-SELECTED CONFIGURATIONS  [R1 points 1, 4]
# =============================================================================
_draw_files = sorted(glob.glob(os.path.join(UNITS, "draws_*.csv")))
DRAWS = pd.concat([pd.read_csv(p) for p in _draw_files], ignore_index=True) if _draw_files else pd.DataFrame()
if len(DRAWS):
    g = DRAWS.groupby(["dataset", "inputs", "regime", "detector"])
    DRAW_SUM = g.PR_AUC.agg(mean="mean", sd="std", min="min", max="max", n="count").reset_index()
    ref = DRAWS[DRAWS.draw == 0].set_index(["dataset", "inputs", "regime", "detector"]).PR_AUC
    DRAW_SUM["reference_draw"] = DRAW_SUM.set_index(["dataset", "inputs", "regime", "detector"]).index.map(ref)
    DRAW_SUM["reference_rank_of_n"] = [
        int((DRAWS[(DRAWS.dataset == r.dataset) & (DRAWS.inputs == r.inputs)
                   & (DRAWS.regime == r.regime) & (DRAWS.detector == r.detector)]
             .PR_AUC > r.reference_draw).sum() + 1) for r in DRAW_SUM.itertuples()]
    DRAW_SUM.to_csv(os.path.join(TABLES, "A3_five_draws_summary.csv"), index=False)
    print(DRAW_SUM.round(4).to_string(index=False))
    # the three subsampled detectors must reproduce the R1 subsampling study
    _r1s = pd.read_csv(os.path.join(R1_DIR, "table_subsampling_raw.csv"))
    _r1s = _r1s[_r1s.rule == "random"].rename(columns={"seed": "draw", "PR_AUC": "PR_AUC_r1"})
    _m = DRAWS[DRAWS.inputs == "ref"].merge(_r1s[["dataset", "regime", "detector", "draw", "PR_AUC_r1"]],
                                           on=["dataset", "regime", "detector", "draw"])
    print(f"\nLOF / One-Class SVM / DBSCAN draws vs the R1 subsampling study: {len(_m)} values, "
          f"largest |difference| = {(_m.PR_AUC - _m.PR_AUC_r1).abs().max():.1e}")

# ---- A4 ------------------------------------------------------------------- #
_tf = [os.path.join(UNITS, f"tuned_ensemble_{d}.csv") for d in ("ULB", "PaySim")]
TUNED = pd.concat([pd.read_csv(p) for p in _tf if os.path.exists(p)], ignore_index=True) \
    if any(os.path.exists(p) for p in _tf) else pd.DataFrame()
if len(TUNED):
    TUNED.to_csv(os.path.join(TABLES, "A4_ensemble_at_selected_configurations.csv"), index=False)
    for (ds, rg, st), g in TUNED.groupby(["dataset", "regime", "setting"]):
        g = g.sort_values("test_PR_AUC", ascending=False)
        print(f"  [{ds}/{rg}/{st:<8}] " + " > ".join(f"{r.detector} {r.test_PR_AUC:.4f}"
                                                    for r in g.itertuples()))
    _t18 = {("PaySim", "N", "DBSCAN"): 0.2169, ("PaySim", "N", "IsolationForest"): 0.0687,
            ("PaySim", "N", "KMeans"): 0.1378, ("PaySim", "N", "LOF"): 0.0068,
            ("PaySim", "N", "OneClassSVM"): 0.2164, ("PaySim", "U", "DBSCAN"): 0.2219,
            ("PaySim", "U", "IsolationForest"): 0.0618, ("PaySim", "U", "KMeans"): 0.1438,
            ("PaySim", "U", "LOF"): 0.0069, ("PaySim", "U", "OneClassSVM"): 0.2465,
            ("ULB", "N", "DBSCAN"): 0.0290, ("ULB", "N", "IsolationForest"): 0.0401,
            ("ULB", "N", "KMeans"): 0.1021, ("ULB", "N", "LOF"): 0.6032,
            ("ULB", "N", "OneClassSVM"): 0.1248, ("ULB", "U", "DBSCAN"): 0.0791,
            ("ULB", "U", "IsolationForest"): 0.0518, ("ULB", "U", "KMeans"): 0.0794,
            ("ULB", "U", "LOF"): 0.3175, ("ULB", "U", "OneClassSVM"): 0.0839}
    _sel = TUNED[TUNED.setting == "selected"]
    _dd = [abs(r.test_PR_AUC - _t18[(r.dataset, r.regime, r.detector)])
           for r in _sel.itertuples() if (r.dataset, r.regime, r.detector) in _t18]
    print(f"\nselected members vs Table 18 of R1: {len(_dd)} values, largest |difference| = "
          f"{max(_dd) if _dd else float('nan'):.1e}")


dataset inputs regime        detector   mean     sd    min    max  n  reference_draw  reference_rank_of_n
 PaySim    ref      N          DBSCAN 0.0559 0.0425 0.0247 0.1082  5          0.0248                    4
 PaySim    ref      N        Ensemble 0.1355 0.0430 0.0843 0.1735  5          0.1706                    2
 PaySim    ref      N       Ensemble4 0.0919 0.0105 0.0812 0.1092  5          0.0874                    4
 PaySim    ref      N IsolationForest 0.0165 0.0000 0.0165 0.0165  5          0.0165                    1
 PaySim    ref      N          KMeans 0.0458 0.0000 0.0458 0.0458  5          0.0458                    1
 PaySim    ref      N             LOF 0.0031 0.0001 0.0030 0.0032  5          0.0030                    5
 PaySim    ref      N     OneClassSVM 0.1045 0.0065 0.0994 0.1149  5          0.1018                    3
 PaySim    ref      U          DBSCAN 0.0380 0.0177 0.0248 0.0607  5          0.0248                    5
 PaySim    ref      U        Ensemble 0.1148 0

In [14]:
# =============================================================================
# A5. PaySim TEMPORAL ANALYSES WITHOUT step: Table 11 and Table 10 formats
#                                                                [R1 point 3]
# =============================================================================
_rf = sorted(glob.glob(os.path.join(UNITS, "PaySim_rolling_nostep_fold*.csv")))
if _rf:
    d = pd.concat([pd.read_csv(p) for p in _rf], ignore_index=True)
    d = d[d.metric == "PR_AUC"].copy()
    d["prevalence"] = d.n_pos_test / d.n_test
    d["lift"] = d.value / d.prevalence
    d[["fold", "regime", "detector", "value", "prevalence", "lift", "n_test", "n_pos_test"]] \
        .rename(columns={"value": "PR_AUC"}) \
        .to_csv(os.path.join(TABLES, "A5_rolling_nostep_per_origin.csv"), index=False)
    ROLL = (d.groupby(["regime", "detector"])
              .agg(mean_PR_AUC=("value", "mean"), SD=("value", "std"),
                   min=("value", "min"), max=("value", "max"),
                   mean_lift=("lift", "mean"), origins=("value", "count"))
              .reset_index())
    ROLL["CV"] = ROLL.SD / ROLL.mean_PR_AUC
    ROLL.to_csv(os.path.join(TABLES, "A5_rolling_nostep_summary.csv"), index=False)
    print("PaySim rolling origins WITHOUT step (Table 11 format):")
    print(ROLL.round(4).to_string(index=False))
    print("\nper origin (PR-AUC, normal-only regime):")
    print(d[d.regime == "N"].pivot_table(index="detector", columns="fold", values="value").round(4).to_string())
    print("fold prevalence:", d.groupby("fold").prevalence.first().round(5).to_dict())

_parts = {"chronological": "PaySim_nostep", "random": "PaySim_random_nostep"}
_frames = []
for arm, unit in _parts.items():
    m = unit_metrics(unit)
    if m is not None:
        _frames.append(m[m.metric == "PR_AUC"].assign(arm=arm))
_mf = [unit_metrics(f"PaySim_matched_nostep_draw{s}") for s in range(3)]
_mf = [m[m.metric == "PR_AUC"] for m in _mf if m is not None]
if len(_frames) == 2 and len(_mf) == 3:
    piv = pd.concat(_frames).pivot_table(index=["regime", "detector"], columns="arm", values="value")
    matched = pd.concat(_mf).groupby(["regime", "detector"]).value.agg(["mean", "min", "max"])
    SPLIT = piv.join(matched.rename(columns={"mean": "random_matched", "min": "matched_min",
                                             "max": "matched_max"})).reset_index()
    SPLIT["gap_total"] = SPLIT.random - SPLIT.chronological
    SPLIT["gap_prevalence"] = SPLIT.random - SPLIT.random_matched
    SPLIT["gap_temporal"] = SPLIT.random_matched - SPLIT.chronological
    SPLIT.to_csv(os.path.join(TABLES, "A5_split_decomposition_nostep.csv"), index=False)
    print("\nPaySim random vs chronological WITHOUT step (Table 10 format):")
    print(SPLIT.round(4).to_string(index=False))
else:
    print("split comparison without step: incomplete, skipped")


PaySim rolling origins WITHOUT step (Table 11 format):
regime        detector  mean_PR_AUC     SD    min    max  mean_lift  origins     CV
     N          DBSCAN       0.0189 0.0240 0.0016 0.0539    20.0532        5 1.2721
     N        Ensemble       0.0528 0.0842 0.0055 0.2019    25.0510        5 1.5959
     N IsolationForest       0.0082 0.0135 0.0016 0.0323     3.6014        5 1.6451
     N          KMeans       0.0177 0.0287 0.0028 0.0690     8.0211        5 1.6237
     N             LOF       0.1373 0.1708 0.0067 0.4198    95.9157        5 1.2446
     N     OneClassSVM       0.0707 0.1101 0.0047 0.2647    35.2479        5 1.5580
     U          DBSCAN       0.0094 0.0139 0.0016 0.0341     4.8610        5 1.4711
     U        Ensemble       0.0453 0.0774 0.0051 0.1834    18.6900        5 1.7067
     U IsolationForest       0.0095 0.0163 0.0016 0.0386     3.8162        5 1.7176
     U          KMeans       0.0176 0.0286 0.0028 0.0688     8.0023        5 1.6233
     U             LO

In [15]:
# =============================================================================
# A6. DIAGNOSTIC OF THE META-CLASSIFIER COLLAPSE (ULB, regime N)  [R1 point 5]
#     Re-draws the label-efficiency study of R1 (Section 4.5, Figure 5) from the
#     saved scores, keeps every fitted coefficient, and asks why the collapse
#     rate rises with the labelling budget.
# =============================================================================
from scipy.stats import kstest, ks_2samp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROW_BUDGETS = [500, 1000, 2500, 5000, 10000, 25000, None]   # as in R1 (cell 9)
N_BUDGET_SEEDS = 20
COLLAPSE = 0.05

z = load_scores("headline_ULB")
yva, yte = z["__yva"], z["__yte"]
cal = {m: ValidationCalibrator(z[f"N|{m}|val"]) for m in MEMBERS5}
Fva = np.column_stack([cal[m].ecdf(z[f"N|{m}|val"]) for m in MEMBERS5])
Fte = np.column_stack([cal[m].ecdf(z[f"N|{m}|test"]) for m in MEMBERS5])
full = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Fva, yva)
w_full = full.coef_[0]

rows = []
for budget in ROW_BUDGETS:
    for s in range(1 if budget is None else N_BUDGET_SEEDS):
        rs = np.random.default_rng(SEED + s)
        idx = np.arange(len(yva)) if budget is None else rs.choice(len(yva), budget, replace=False)
        nf = int(yva[idx].sum())
        r = {"budget": budget or len(yva), "seed": s, "n_frauds": nf}
        if nf < 2:
            rows.append({**r, "valid": 0}); continue
        rec = {"valid": 1}
        n_lab = budget or len(yva)
        # as run (C = 1): the penalty's weight relative to the data term falls as 1/n.
        # penalty_held: C = 2500/n keeps that relative weight at its 2,500-transaction level.
        for mode, C_ in (("as_run", 1.0), ("penalty_held", 2500.0 / n_lab)):
            clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=C_).fit(Fva[idx], yva[idx])
            ap = average_precision_score(yte, clf.predict_proba(Fte)[:, 1])
            rec[f"PR_AUC_{mode}"] = ap
            if mode == "as_run":
                w = clf.coef_[0]
                rec.update({f"coef_{m}": c for m, c in zip(MEMBERS5, w)})
                rec["intercept"] = float(clf.intercept_[0])
                rec["cosine_to_full_validation_fit"] = float(w @ w_full / (np.linalg.norm(w) * np.linalg.norm(w_full)))
        rows.append({**r, **rec})
META = pd.DataFrame(rows)
META["collapsed"] = np.where(META.valid == 1, (META.PR_AUC_as_run < COLLAPSE).astype(float), np.nan)
META["collapsed_penalty_held"] = np.where(META.valid == 1, (META.PR_AUC_penalty_held < COLLAPSE).astype(float), np.nan)
META.to_csv(os.path.join(TABLES, "A6_meta_draws.csv"), index=False)

_r1le = pd.read_csv(os.path.join(R1_DIR, "table_label_efficiency_raw.csv"))
_m = META.merge(_r1le, on=["budget", "seed"])
print(f"draws vs R1 label-efficiency study: largest |difference| = "
      f"{np.nanmax(np.abs(_m.PR_AUC_as_run - _m.PR_AUC)):.1e}")

coef_cols = [f"coef_{m}" for m in MEMBERS5]
ok = META[META.valid == 1]
BY_BUDGET = ok.groupby("budget").agg(
    valid_draws=("seed", "count"), collapse_rate=("collapsed", "mean"),
    collapse_rate_penalty_held=("collapsed_penalty_held", "mean"),
    median_PR_AUC_penalty_held=("PR_AUC_penalty_held", "median"),
    median_PR_AUC_not_collapsed=("PR_AUC_as_run", lambda v: np.median(v[v >= COLLAPSE]) if np.any(v >= COLLAPSE) else np.nan),
    median_cosine_to_full_fit=("cosine_to_full_validation_fit", "median"),
    **{c.replace("coef_", "median_coef_"): (c, "median") for c in coef_cols}).reset_index()
BY_BUDGET.to_csv(os.path.join(TABLES, "A6_meta_by_budget.csv"), index=False)
BY_STATUS = ok.groupby("collapsed")[coef_cols + ["intercept", "cosine_to_full_validation_fit"]].mean().reset_index()
BY_STATUS.to_csv(os.path.join(TABLES, "A6_meta_by_collapse_status.csv"), index=False)

# regularisation path on the full validation set (diagnostic; nothing is selected on test)
path = []
for C_ in [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]:
    g = LogisticRegression(max_iter=5000, class_weight="balanced", C=C_).fit(Fva, yva)
    path.append({"C": C_, "val_PR_AUC_in_sample": average_precision_score(yva, g.predict_proba(Fva)[:, 1]),
                 "test_PR_AUC": average_precision_score(yte, g.predict_proba(Fte)[:, 1]),
                 **{f"coef_{m}": c for m, c in zip(MEMBERS5, g.coef_[0])}})
PATH = pd.DataFrame(path); PATH.to_csv(os.path.join(TABLES, "A6_meta_regularisation_path.csv"), index=False)

# validation-to-test shift of the ECDF features
shift = []
for j, m in enumerate(MEMBERS5):
    ks_f = ks_2samp(Fte[yte == 1, j], Fva[yva == 1, j])
    shift.append({"member": m,
                  "KS_test_legit_vs_uniform": kstest(Fte[yte == 0, j], "uniform").statistic,
                  "mean_ECDF_test_legit": Fte[yte == 0, j].mean(),
                  "mean_ECDF_val_fraud": Fva[yva == 1, j].mean(),
                  "mean_ECDF_test_fraud": Fte[yte == 1, j].mean(),
                  "KS_fraud_val_vs_test": ks_f.statistic, "p_fraud_val_vs_test": ks_f.pvalue,
                  "share_test_fraud_above_val_max": float(np.mean(Fte[yte == 1, j] >= 1.0)),
                  "share_test_legit_above_val_max": float(np.mean(Fte[yte == 0, j] >= 1.0)),
                  "PR_AUC_val_member": average_precision_score(yva, Fva[:, j]),
                  "PR_AUC_test_member": average_precision_score(yte, Fte[:, j])})
SHIFT = pd.DataFrame(shift); SHIFT.to_csv(os.path.join(TABLES, "A6_meta_feature_shift.csv"), index=False)

# the same objective fitted on the TEST labels: what log-loss itself rewards (diagnostic only)
o = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Fte, yte)
OBJ = pd.DataFrame([{"fit": "full validation (as run)",
                     "PR_AUC_on_validation": average_precision_score(yva, full.predict_proba(Fva)[:, 1]),
                     "PR_AUC_on_test": average_precision_score(yte, full.predict_proba(Fte)[:, 1]),
                     **{f"coef_{m}": c for m, c in zip(MEMBERS5, full.coef_[0])}},
                    {"fit": "test labels (diagnostic only, in-sample)",
                     "PR_AUC_on_validation": average_precision_score(yva, o.predict_proba(Fva)[:, 1]),
                     "PR_AUC_on_test": average_precision_score(yte, o.predict_proba(Fte)[:, 1]),
                     **{f"coef_{m}": c for m, c in zip(MEMBERS5, o.coef_[0])}}])
OBJ.to_csv(os.path.join(TABLES, "A6_meta_objective_check.csv"), index=False)

pd.set_option("display.width", 220)
print("\nby budget:\n", BY_BUDGET.round(3).to_string(index=False))
print("\nmean coefficients by collapse status:\n", BY_STATUS.round(2).to_string(index=False))
print("\nregularisation path (full validation set):\n", PATH.round(4).to_string(index=False))
print("\nECDF feature shift, validation -> test:\n", SHIFT.round(4).to_string(index=False))
print("\nobjective check:\n", OBJ.round(4).to_string(index=False))

# figure (supplementary): coefficient paths and collapse rate
fig, ax = plt.subplots(1, 2, figsize=(7.0, 2.8))
x = BY_BUDGET.budget.values
for m in MEMBERS5:
    ax[0].plot(x, BY_BUDGET[f"median_coef_{m}"], marker="o", ms=3, lw=1.2, label=m)
ax[0].axhline(0, c="grey", lw=0.6)
ax[0].set_xscale("log"); ax[0].set_xlabel("Labelled transactions"); ax[0].set_ylabel("Median coefficient")
ax[0].legend(fontsize=6.5, frameon=False); ax[0].set_title("(a) Coefficient paths", fontsize=8.5)
ax[1].plot(x, BY_BUDGET.collapse_rate, marker="o", ms=3, lw=1.2, label="as run (C = 1)")
ax[1].plot(x, BY_BUDGET.collapse_rate_penalty_held, marker="s", ms=3, lw=1.2, ls="--",
           label="relative penalty held (C = 2,500/n)")
ax[1].set_xscale("log"); ax[1].set_xlabel("Labelled transactions"); ax[1].set_ylabel("Share of draws with PR-AUC < 0.05")
ax[1].legend(fontsize=6.5, frameon=False); ax[1].set_title("(b) Collapse rate", fontsize=8.5)
for a in ax:
    a.tick_params(labelsize=7); a.xaxis.label.set_size(7.5); a.yaxis.label.set_size(7.5)
fig.tight_layout()
fig.savefig(os.path.join(FIGS, "meta_classifier_diagnostic.png"), dpi=600)
fig.savefig(os.path.join(FIGS, "meta_classifier_diagnostic.pdf"))
plt.close(fig)
print("figure ->", os.path.join(FIGS, "meta_classifier_diagnostic.png"))


draws vs R1 label-efficiency study: largest |difference| = 1.5e-08

by budget:
  budget  valid_draws  collapse_rate  collapse_rate_penalty_held  median_PR_AUC_penalty_held  median_PR_AUC_not_collapsed  median_cosine_to_full_fit  median_coef_IsolationForest  median_coef_LOF  median_coef_OneClassSVM  median_coef_DBSCAN  median_coef_KMeans
    500            4          0.250                       0.250                       0.160                        0.158                      0.335                        2.229            4.684                    3.355               2.765               2.606
   1000            4          0.000                       0.000                       0.164                        0.176                      0.485                        3.051            5.761                    3.661               1.178               3.829
   2500           15          0.133                       0.133                       0.162                        0.164                      0

In [16]:
# =============================================================================
# Z. SUMMARY FILE  --  every value the revised manuscript will quote
# =============================================================================
_lines = [f"R2 analyses, ARRAY-D-26-04414R1 -- written {pd.Timestamp.now().isoformat()}",
          f"output folder: {OUT_DIR}", ""]
_rc = os.path.join(OUT_DIR, "reproduction_check.json")
if os.path.exists(_rc):
    _lines += ["== Reproduction of the R1 headline run ==", open(_rc).read(), ""]
_units = sorted(os.path.basename(p)[:-5] for p in glob.glob(os.path.join(UNITS, "*.done")))
_lines += ["== Units completed ==", ", ".join(_units), ""]
for p in sorted(glob.glob(os.path.join(TABLES, "*.csv"))):
    t = pd.read_csv(p)
    _lines += [f"== {os.path.basename(p)} ==",
               t.to_string(index=False, float_format=lambda v: f"{v:.6g}"), ""]
with open(os.path.join(OUT_DIR, "R2_VALUES.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(_lines))
print(f"{len(_units)} units completed; summary -> {os.path.join(OUT_DIR, 'R2_VALUES.txt')}")
_expected = (["headline_ULB", "headline_PaySim", "PaySim_noresid", "PaySim_nostep", "PaySim_noboth",
              "PaySim_random_nostep", "tuned_ensemble_ULB", "tuned_ensemble_PaySim"]
             + [f"PaySim_rolling_nostep_fold{k}" for k in range(5)]
             + [f"PaySim_matched_nostep_draw{s}" for s in range(3)]
             + [f"draws_{d}_ref_{r}" for d in ("ULB", "PaySim") for r in ("U", "N")])
_missing = [u for u in _expected if u not in _units]
print("all required units done" if not _missing else f"MISSING: {_missing}")


20 units completed; summary -> C:\Users\LENOVO\Desktop\Unsupervised and Semi-Supervised Anomaly Detection for Credit Card Fraud\2_Revision_R2\resultats_R2\R2_VALUES.txt
all required units done


In [17]:
assert all(k in globals() for k in ("SELECTED_R1", "reset_inputs", "DETECTORS", "UNITS", "TABLES",
                                    "is_done", "stamp", "mark_done", "reference_set", "ecdf_rank_average")), \
    "Noyau redemarre : les definitions du notebook ne sont plus en memoire. Executez d'abord tout le notebook."

import os, json, glob
SELECTED_R1[("PaySim", "U", "IsolationForest")] = {"n_estimators": 200}   # [R2 fix] R1 selection file
R1_FIXED_ENSEMBLE_ON_WINDOW = {("PaySim", "U"): 0.2426, ("PaySim", "N"): 0.2403}

TUNED_UNIT = {"ULB": "tuned_ensemble_ULB", "PaySim": "tuned_ensemble_PaySim_v2"}


def tuned_ensemble(ds):
    unit = TUNED_UNIT[ds]
    if is_done(unit):
        stamp(f"{unit}: already done, skipped"); return
    stamp(f"{unit}: start")
    reset_inputs()
    sp = chronological_split(ds)
    if len(sp.Xte) > GRID_EVAL_CAP:
        wv, wt = slice(-GRID_EVAL_CAP, None), slice(-GRID_EVAL_CAP, None)
        window = f"most recent {GRID_EVAL_CAP:,} validation and test rows"
    else:
        wv, wt = slice(None), slice(None)
        window = "full validation and test windows"
    Xte_w, yte_w, yva_w = sp.Xte[wt], sp.yte[wt], sp.yva[wv]
    rows = []
    for regime in ("U", "N"):
        Xref = reference_set(sp, regime)
        rho = rho_for(sp, regime, RHO_DEFAULT)
        for setting in ("fixed", "selected"):
            sc = {}
            for det in MEMBERS5:
                cfg = SELECTED_R1[(ds, regime, det)] if setting == "selected" else {}
                kw = dict(cfg)
                if det == "OneClassSVM":
                    kw.setdefault("nu", rho)
                sc[det] = DETECTORS[det](Xref, sp.Xva, Xte_w, seed=SEED, **kw)
                rows.append({"dataset": ds, "regime": regime, "setting": setting,
                             "detector": det, "config": json.dumps(cfg),
                             "val_PR_AUC": average_precision_score(yva_w, sc[det].val[wv]),
                             "test_PR_AUC": average_precision_score(yte_w, sc[det].test),
                             "window": window})
            cals = {m: ValidationCalibrator(sc[m].val) for m in MEMBERS5}   # full validation window
            for name, mem in (("Ensemble", MEMBERS5), (ENS4, MEMBERS4)):
                ev = ecdf_rank_average({m: cals[m] for m in mem}, {m: sc[m].val for m in mem})
                et = ecdf_rank_average({m: cals[m] for m in mem}, {m: sc[m].test for m in mem})
                rows.append({"dataset": ds, "regime": regime, "setting": setting,
                             "detector": name, "config": f"members at the {setting} configuration",
                             "val_PR_AUC": average_precision_score(yva_w, ev[wv]),
                             "test_PR_AUC": average_precision_score(yte_w, et),
                             "window": window})
            stamp(f"  {ds}/{regime}/{setting}: " + " | ".join(
                f"{r['detector']} {r['test_PR_AUC']:.4f}" for r in rows[-7:]))
            if setting == "fixed" and (ds, regime) in R1_FIXED_ENSEMBLE_ON_WINDOW:
                v = [r["test_PR_AUC"] for r in rows if r["dataset"] == ds and r["regime"] == regime
                     and r["setting"] == "fixed" and r["detector"] == "Ensemble"][0]
                ref = R1_FIXED_ENSEMBLE_ON_WINDOW[(ds, regime)]
                print(f"    fixed-configuration ensemble on the window: {v:.4f} (R1: {ref})"
                      + ("" if abs(v - ref) <= 0.0000501 else "   <-- differs from R1"))
    pd.DataFrame(rows).to_csv(os.path.join(UNITS, unit + ".csv"), index=False)
    mark_done(unit)
    stamp(f"{unit}: done")


tuned_ensemble("PaySim")

# ---- A4 ------------------------------------------------------------------- #
_tf = [os.path.join(UNITS, f"{TUNED_UNIT[d]}.csv") for d in ("ULB", "PaySim")]
TUNED = pd.concat([pd.read_csv(p) for p in _tf if os.path.exists(p)], ignore_index=True) \
    if any(os.path.exists(p) for p in _tf) else pd.DataFrame()
if len(TUNED):
    TUNED.to_csv(os.path.join(TABLES, "A4_ensemble_at_selected_configurations.csv"), index=False)
    for (ds, rg, st), g in TUNED.groupby(["dataset", "regime", "setting"]):
        g = g.sort_values("test_PR_AUC", ascending=False)
        print(f"  [{ds}/{rg}/{st:<8}] " + " > ".join(f"{r.detector} {r.test_PR_AUC:.4f}"
                                                    for r in g.itertuples()))

# ---- Z. summary file ------------------------------------------------------ #
_lines = [f"R2 analyses, ARRAY-D-26-04414R1 -- written {pd.Timestamp.now().isoformat()}",
          f"output folder: {OUT_DIR}", ""]
_rc = os.path.join(OUT_DIR, "reproduction_check.json")
if os.path.exists(_rc):
    _lines += ["== Reproduction of the R1 headline run ==", open(_rc).read(), ""]
_units = sorted(os.path.basename(p)[:-5] for p in glob.glob(os.path.join(UNITS, "*.done")))
_lines += ["== Units completed ==", ", ".join(_units), ""]
for p in sorted(glob.glob(os.path.join(TABLES, "*.csv"))):
    t = pd.read_csv(p)
    _lines += [f"== {os.path.basename(p)} ==",
               t.to_string(index=False, float_format=lambda v: f"{v:.6g}"), ""]
with open(os.path.join(OUT_DIR, "R2_VALUES.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(_lines))
print("done:", "tuned_ensemble_PaySim_v2" in _units)

[12:40:14  + 153.4 min] tuned_ensemble_PaySim_v2: start
  raw cache cleared
[12:47:23  + 160.5 min]   PaySim/U/fixed: IsolationForest 0.0644 | LOF 0.0076 | OneClassSVM 0.2437 | DBSCAN 0.0665 | KMeans 0.1121 | Ensemble 0.2426 | Ensemble4 0.1142
    fixed-configuration ensemble on the window: 0.2426 (R1: 0.2426)
[12:57:36  + 170.7 min]   PaySim/U/selected: IsolationForest 0.0644 | LOF 0.0069 | OneClassSVM 0.2465 | DBSCAN 0.2219 | KMeans 0.1438 | Ensemble 0.0682 | Ensemble4 0.0669
[13:04:41  + 177.8 min]   PaySim/N/fixed: IsolationForest 0.0687 | LOF 0.0075 | OneClassSVM 0.2185 | DBSCAN 0.0665 | KMeans 0.1129 | Ensemble 0.2403 | Ensemble4 0.1109
    fixed-configuration ensemble on the window: 0.2403 (R1: 0.2403)
[13:14:42  + 187.8 min]   PaySim/N/selected: IsolationForest 0.0687 | LOF 0.0068 | OneClassSVM 0.2164 | DBSCAN 0.2169 | KMeans 0.1378 | Ensemble 0.0708 | Ensemble4 0.0695
[13:14:42  + 187.8 min] tuned_ensemble_PaySim_v2: done
  [PaySim/N/fixed   ] Ensemble 0.2403 > OneClassSVM 0.2